# Goal 3 — Human perceptual correspondence  
## Stage 1: annotation ingestion, interval reconstruction, and support audit

This notebook reconstructs the human annotation reference from the four rater folders and **stops before correspondence modeling**.

Primary safeguards:

- the recording, not the 5-second interface row, is the analytical unit;
- all JSON item intervals are parsed and audited;
- exact duplicates are removed;
- overlapping and adjacent intervals are unioned only after a merge-tolerance sensitivity audit;
- repeated annotations of the same recording by different raters are preserved and flagged;
- production annotations are merged to the frozen 519-recording population;
- family/item prevalence and rater allocation are audited before any model is permitted;
- no human label is treated as physical ground truth or as analytical validation of a signal-derived indicator.


In [ ]:
from __future__ import annotations

import ast
import hashlib
import json
import math
import os
import platform
import re
import sys
import warnings
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 220)

# ------------------------------------------------------------------
# Configurable project paths
# ------------------------------------------------------------------
def locate_analysis_root() -> Path:
    configured = os.environ.get("PAPER1_ANALYSIS_ROOT", "").strip()
    if configured:
        return Path(configured).expanduser().resolve()
    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").is_file() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the analysis repository root")

ANALYSIS_ROOT = locate_analysis_root()
PAPER_ROOT = ANALYSIS_ROOT.parent
PIPELINE_ROOT = Path(os.environ.get("PAPER1_PIPELINE_ROOT", PAPER_ROOT / "Paper_1_Final_Cleanup"))
HUMAN_QC_ROOT = Path(os.environ.get("PAPER1_HUMAN_QC_ROOT", PAPER_ROOT.parent / "Data_13072026" / "Bamboo_passage_HumanQC"))
METADATA_ROOT = Path(os.environ.get("PAPER1_DATA_FREEZE_DIR", PIPELINE_ROOT / "outputs" / "data"))
FAMILY_FREEZE_ROOT = Path(os.environ.get("PAPER1_FAMILY_FREEZE_DIR", PIPELINE_ROOT / "outputs" / "features"))

OUTPUT_ROOT = ANALYSIS_ROOT / "outputs" / "goal3_human_correspondence" / "stage1_annotation_audit"
TABLE_DIR = OUTPUT_ROOT / "tables"
FIGURE_DIR = OUTPUT_ROOT / "figures"
SOURCE_DIR = FIGURE_DIR / "source_data"
LOG_DIR = OUTPUT_ROOT / "logs"
MANIFEST_DIR = OUTPUT_ROOT / "manifest"

for directory in [TABLE_DIR, FIGURE_DIR, SOURCE_DIR, LOG_DIR, MANIFEST_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# Frozen Stage-1 settings
# ------------------------------------------------------------------
EXPECTED_RATERS = ["Abbas", "Liya", "Samaana", "Samara"]
SEGMENT_BOUNDARY_TOLERANCE_SEC = 0.050
PROVISIONAL_MERGE_TOLERANCE_SEC = 0.010
MERGE_SENSITIVITY_TOLERANCES_SEC = [0.0, 0.001, 0.010, 0.050, 0.100]
FINAL_POPULATION_N = 519

# This notebook is deliberately an audit-only stage.
ALLOW_CORRESPONDENCE_MODELS = False

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Human-QC root:", HUMAN_QC_ROOT)
print("Output root:", OUTPUT_ROOT)


## 1. Annotation schema and construct crosswalk

The eight GUI columns are preserved exactly. Seven contain acquisition-related percepts and one contains task-adherence events.

The six signal-family outcomes are reconstructed as follows:

- **QADD:** union of `Environmental noise` and `Competing speech`;
- **QGAIN:** `Volume unstable`;
- **QREV:** `Reverberation/echo`;
- **QCHAN:** `Platform effects`;
- **QDIST:** `Clipping`;
- **QTEMP:** `Temporal discontinuities`.

`Any non-task related content` is retained separately as a task-adherence/negative-control family.


In [ ]:
ANNOTATION_SCHEMA = {
    "Environmental noise": [
        "Beep",
        "HVAC",
        "Microphone rubbing",
        "Non-specific environmental noise",
        "Pets",
        "TV (non-speech)",
        "Traffic",
    ],
    "Any non-task related content": [
        "Coughing",
        "Extra or filler word",
        "Laughing",
        "Lip smacking/mouth sounds",
        "Missed word",
        "Mouse clicking/keyboard noise",
    ],
    "Competing speech": [
        "Other human speakers",
        "TV (speech)",
    ],
    "Volume unstable": [
        "Volume changes",
        "Volume too loud",
        "Volume too quiet",
    ],
    "Clipping": [
        "Crackling on loud syllables",
        "Grainy, sandy, buzzing voice texture",
        "Whole voice sounds crushed or overloaded",
    ],
    "Reverberation/echo": [
        "Echo",
        "Reverb",
    ],
    "Platform effects": [
        "Muffled, filtered, telephone-like",
        "Underwater, robotic, warbly, codec-like",
    ],
    "Temporal discontinuities": [
        "Audio glitching",
        "Audio lagging",
        "Audio skipped/missed",
    ],
}

ANNOTATION_COLUMNS = list(ANNOTATION_SCHEMA)
BASE_COLUMNS = [
    "file_name",
    "source_path",
    "segment_number",
    "segment_type",
    "pause_position",
    "onset_seconds_absolute",
    "offset_seconds_absolute",
    "duration_seconds",
]

SOURCE_TO_SIGNAL_FAMILY = {
    "Environmental noise": "QADD",
    "Competing speech": "QADD",
    "Volume unstable": "QGAIN",
    "Reverberation/echo": "QREV",
    "Platform effects": "QCHAN",
    "Clipping": "QDIST",
    "Temporal discontinuities": "QTEMP",
    "Any non-task related content": "TASK",
}

ITEM_TO_SOURCE_FAMILY = {
    item: family
    for family, items in ANNOTATION_SCHEMA.items()
    for item in items
}

# Human outcomes proposed for the later, construct-matched model stage.
PROPOSED_HUMAN_OUTCOMES = [
    ("QADD", "signal_family", "QADD", "Additive interference: environmental noise or competing speech"),
    ("QADD", "source_family", "Environmental noise", "Environmental noise"),
    ("QADD", "source_family", "Competing speech", "Competing speech"),
    ("QADD", "item", "HVAC", "HVAC"),
    ("QGAIN", "signal_family", "QGAIN", "Volume instability"),
    ("QGAIN", "item", "Volume changes", "Volume changes"),
    ("QGAIN", "item", "Volume too loud", "Volume too loud"),
    ("QGAIN", "item", "Volume too quiet", "Volume too quiet"),
    ("QREV", "signal_family", "QREV", "Reverberation or echo"),
    ("QREV", "item", "Echo", "Echo"),
    ("QREV", "item", "Reverb", "Reverb"),
    ("QCHAN", "signal_family", "QCHAN", "Platform-related degradation"),
    ("QCHAN", "item", "Muffled, filtered, telephone-like", "Muffled or bandwidth-limited"),
    ("QCHAN", "item", "Underwater, robotic, warbly, codec-like", "Codec-like or robotic"),
    ("QDIST", "signal_family", "QDIST", "Clipping or overload"),
    ("QTEMP", "signal_family", "QTEMP", "Temporal discontinuity"),
    ("QTEMP", "item", "Audio glitching", "Audio glitching"),
    ("QTEMP", "item", "Audio lagging", "Audio lagging"),
    ("QTEMP", "item", "Audio skipped/missed", "Audio skipped or missed"),
]

schema_rows = []
for family, items in ANNOTATION_SCHEMA.items():
    for item in items:
        schema_rows.append({
            "source_family": family,
            "signal_family": SOURCE_TO_SIGNAL_FAMILY[family],
            "item": item,
        })
annotation_taxonomy = pd.DataFrame(schema_rows)
display(annotation_taxonomy)


## 2. Utility functions

In [ ]:
def canonical_recording_id(value: Any) -> str:
    """Canonical case-insensitive recording stem for cross-table matching."""
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return ""
    text = str(value).strip().replace("\\", "/")
    name = text.rsplit("/", 1)[-1]
    lower_name = name.lower()
    for suffix in [
        "_segments.csv",
        ".segments.csv",
        ".csv",
        ".wav",
        ".mp3",
        ".m4a",
        ".mp4",
        ".flac",
    ]:
        if lower_name.endswith(suffix):
            name = name[: -len(suffix)]
            lower_name = name.lower()
            break
    return re.sub(r"\s+", "", name).casefold()


def sha256_file(path: Path, block_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            chunk = handle.read(block_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def export_dataframe(frame: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False)


def infer_rater(path: Path) -> str:
    relative_parts = [part.casefold() for part in path.relative_to(HUMAN_QC_ROOT).parts]
    for expected in EXPECTED_RATERS:
        if expected.casefold() in relative_parts:
            return expected

    # Fallback for unusual nesting.
    joined = " ".join(relative_parts)
    for expected in EXPECTED_RATERS:
        if expected.casefold() in joined:
            return expected

    return path.relative_to(HUMAN_QC_ROOT).parts[0] if path.relative_to(HUMAN_QC_ROOT).parts else "UNKNOWN"


def parse_json_dict(value: Any) -> tuple[dict[str, Any], str | None]:
    """Parse an annotation JSON object without silently discarding malformed cells."""
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return {}, None

    text = str(value).strip()
    if text in {"", "{}", "nan", "None"}:
        return {}, None

    try:
        parsed = json.loads(text)
    except Exception:
        try:
            parsed = ast.literal_eval(text)
        except Exception as exc:
            return {}, f"{type(exc).__name__}: {exc}"

    if not isinstance(parsed, dict):
        return {}, f"Expected dict, received {type(parsed).__name__}"

    return parsed, None


def merge_intervals(
    intervals: Iterable[tuple[float, float]],
    tolerance_sec: float,
) -> list[tuple[float, float]]:
    """Union overlapping or near-adjacent intervals."""
    cleaned = sorted(
        {
            (float(start), float(end))
            for start, end in intervals
            if np.isfinite(start) and np.isfinite(end) and end > start
        }
    )
    if not cleaned:
        return []

    merged: list[list[float]] = [[cleaned[0][0], cleaned[0][1]]]
    for start, end in cleaned[1:]:
        previous = merged[-1]
        if start <= previous[1] + tolerance_sec:
            previous[1] = max(previous[1], end)
        else:
            merged.append([start, end])

    return [(float(start), float(end)) for start, end in merged]


def summarize_intervals(intervals: list[tuple[float, float]]) -> dict[str, float | int]:
    if not intervals:
        return {
            "event_count": 0,
            "union_duration_sec": 0.0,
            "first_onset_sec": np.nan,
            "last_offset_sec": np.nan,
        }

    return {
        "event_count": len(intervals),
        "union_duration_sec": float(sum(end - start for start, end in intervals)),
        "first_onset_sec": float(min(start for start, _ in intervals)),
        "last_offset_sec": float(max(end for _, end in intervals)),
    }


def read_tabular(path: Path, usecols: list[str] | None = None) -> pd.DataFrame:
    suffix = path.suffix.casefold()
    if suffix == ".csv":
        return pd.read_csv(path, usecols=usecols, low_memory=False)
    if suffix in {".parquet", ".pq"}:
        frame = pd.read_parquet(path)
        return frame if usecols is None else frame[usecols]
    raise ValueError(f"Unsupported table type: {path}")


ID_COLUMN_CANDIDATES = [
    "recording_id",
    "recording_uid",
    "file_id",
    "file_name",
    "filename",
    "recording_filename",
    "audio_file",
    "audio_filename",
    "wav_file",
]

PARTICIPANT_COLUMN_CANDIDATES = [
    "participant_id",
    "subject_id",
    "participant",
    "subject",
    "subject_uid",
]

COHORT_COLUMN_CANDIDATES = [
    "cohort",
    "group",
    "diagnosis",
    "diagnostic_group",
]


## 3. Discover and inventory all rater files

In [ ]:
if not HUMAN_QC_ROOT.exists():
    raise FileNotFoundError(f"Human-QC root not found: {HUMAN_QC_ROOT}")

csv_paths = sorted(HUMAN_QC_ROOT.rglob("*_segments.csv"))
if not csv_paths:
    csv_paths = sorted(HUMAN_QC_ROOT.rglob("*.csv"))

if not csv_paths:
    raise FileNotFoundError(f"No CSV files found under {HUMAN_QC_ROOT}")

inventory_rows = []
for path in csv_paths:
    stat = path.stat()
    inventory_rows.append({
        "path": str(path),
        "relative_path": str(path.relative_to(HUMAN_QC_ROOT)),
        "rater": infer_rater(path),
        "file_size_bytes": stat.st_size,
        "modified_utc": datetime.fromtimestamp(stat.st_mtime, tz=timezone.utc).isoformat(),
        "sha256": sha256_file(path),
    })

file_inventory = pd.DataFrame(inventory_rows)
folder_summary = (
    file_inventory.groupby("rater", dropna=False)
    .agg(
        csv_files=("path", "size"),
        total_bytes=("file_size_bytes", "sum"),
    )
    .reset_index()
    .sort_values("rater")
)

display(folder_summary)
export_dataframe(file_inventory, TABLE_DIR / "humanqc_file_inventory.csv")
export_dataframe(folder_summary, TABLE_DIR / "humanqc_folder_summary.csv")

missing_expected_raters = sorted(set(EXPECTED_RATERS) - set(folder_summary["rater"]))
unexpected_raters = sorted(set(folder_summary["rater"]) - set(EXPECTED_RATERS))
print("Missing expected rater folders:", missing_expected_raters)
print("Unexpected/unknown rater folders:", unexpected_raters)


## 4. Load files, audit schemas, and reconstruct recording identities

In [ ]:
raw_frames: list[pd.DataFrame] = []
schema_audit_rows: list[dict[str, Any]] = []
file_load_rows: list[dict[str, Any]] = []

for csv_path in csv_paths:
    rater = infer_rater(csv_path)

    try:
        frame = pd.read_csv(csv_path, low_memory=False)
        load_error = None
    except Exception as exc:
        frame = pd.DataFrame()
        load_error = f"{type(exc).__name__}: {exc}"

    present_columns = set(frame.columns)
    required_columns = set(BASE_COLUMNS + ANNOTATION_COLUMNS)
    missing_columns = sorted(required_columns - present_columns)
    extra_columns = sorted(present_columns - required_columns)

    file_load_rows.append({
        "path": str(csv_path),
        "rater": rater,
        "loaded": load_error is None,
        "load_error": load_error,
        "rows": len(frame),
        "columns": len(frame.columns),
        "missing_required_columns": " | ".join(missing_columns),
        "extra_columns": " | ".join(extra_columns),
    })

    if load_error is not None or missing_columns:
        continue

    frame = frame.copy()
    frame["annotation_csv_path"] = str(csv_path)
    frame["rater"] = rater
    frame["recording_id"] = frame["file_name"].map(canonical_recording_id)

    # One CSV should represent one recording.
    unique_ids = sorted(set(frame["recording_id"]) - {""})
    schema_audit_rows.append({
        "path": str(csv_path),
        "rater": rater,
        "rows": len(frame),
        "unique_recording_ids": len(unique_ids),
        "recording_ids": " | ".join(unique_ids),
        "segment_number_unique": frame["segment_number"].nunique(dropna=True),
        "segment_type_values": " | ".join(sorted(frame["segment_type"].dropna().astype(str).unique())),
        "file_name_values": frame["file_name"].nunique(dropna=True),
        "source_path_values": frame["source_path"].nunique(dropna=True),
    })
    raw_frames.append(frame)

file_load_audit = pd.DataFrame(file_load_rows)
schema_audit = pd.DataFrame(schema_audit_rows)

display(file_load_audit)
display(schema_audit.head())

export_dataframe(file_load_audit, LOG_DIR / "humanqc_file_load_audit.csv")
export_dataframe(schema_audit, LOG_DIR / "humanqc_schema_audit.csv")

if (~file_load_audit["loaded"]).any():
    raise RuntimeError("At least one annotation CSV failed to load. Review humanqc_file_load_audit.csv.")

if file_load_audit["missing_required_columns"].fillna("").ne("").any():
    raise RuntimeError("At least one annotation CSV is missing required columns.")

raw_segments = pd.concat(raw_frames, ignore_index=True)
raw_segments["recording_rating_id"] = (
    raw_segments["recording_id"].astype(str)
    + "::"
    + raw_segments["rater"].astype(str)
)

print("Loaded segment rows:", len(raw_segments))
print("Unique recording IDs:", raw_segments["recording_id"].nunique())
print("Unique recording-rater ratings:", raw_segments["recording_rating_id"].nunique())


## 5. Segment geometry and duplicate-rating audit

In [ ]:
segment_audit_rows = []

for rating_id, group in raw_segments.groupby("recording_rating_id", sort=False):
    ordered = group.sort_values(["onset_seconds_absolute", "segment_number"]).copy()
    onset = pd.to_numeric(ordered["onset_seconds_absolute"], errors="coerce").to_numpy(float)
    offset = pd.to_numeric(ordered["offset_seconds_absolute"], errors="coerce").to_numpy(float)
    duration = pd.to_numeric(ordered["duration_seconds"], errors="coerce").to_numpy(float)

    gaps = onset[1:] - offset[:-1] if len(ordered) > 1 else np.array([], dtype=float)
    segment_audit_rows.append({
        "recording_rating_id": rating_id,
        "recording_id": ordered["recording_id"].iloc[0],
        "rater": ordered["rater"].iloc[0],
        "segment_rows": len(ordered),
        "recording_duration_sec": float(np.nanmax(offset)) if len(offset) else np.nan,
        "segment_numbers_unique": ordered["segment_number"].nunique(dropna=True),
        "segment_numbers_sequential": bool(
            ordered["segment_number"].dropna().astype(int).tolist()
            == list(range(1, len(ordered) + 1))
        ),
        "finite_geometry": bool(np.isfinite(onset).all() and np.isfinite(offset).all()),
        "positive_segment_duration": bool(np.all(offset > onset)),
        "duration_column_matches": bool(
            np.allclose(duration, offset - onset, atol=1e-6, rtol=0, equal_nan=False)
        ),
        "max_absolute_intersegment_gap_sec": float(np.max(np.abs(gaps))) if len(gaps) else 0.0,
        "monotonic_nonoverlapping_segments": bool(np.all(gaps >= -1e-6)) if len(gaps) else True,
    })

segment_geometry_audit = pd.DataFrame(segment_audit_rows)

rating_inventory = (
    raw_segments.groupby(["recording_id", "rater"], as_index=False)
    .agg(
        segment_rows=("segment_number", "size"),
        recording_duration_sec=("offset_seconds_absolute", "max"),
        csv_paths=("annotation_csv_path", lambda values: " | ".join(sorted(set(map(str, values))))),
    )
)

duplicate_recordings = (
    rating_inventory.groupby("recording_id")
    .agg(
        ratings=("rater", "size"),
        raters=("rater", lambda values: " | ".join(sorted(set(values)))),
        csv_paths=("csv_paths", lambda values: " | ".join(values)),
    )
    .reset_index()
)
duplicate_recordings["multiple_raters"] = duplicate_recordings["ratings"] > 1

within_rater_duplicates = (
    rating_inventory.groupby(["recording_id", "rater"])
    .size()
    .reset_index(name="rating_rows")
)
within_rater_duplicates = within_rater_duplicates.loc[
    within_rater_duplicates["rating_rows"] > 1
]

display(segment_geometry_audit.describe(include="all").T)
display(duplicate_recordings.loc[duplicate_recordings["multiple_raters"]].head(20))

export_dataframe(segment_geometry_audit, LOG_DIR / "humanqc_segment_geometry_audit.csv")
export_dataframe(duplicate_recordings, TABLE_DIR / "humanqc_duplicate_recording_audit.csv")
export_dataframe(within_rater_duplicates, LOG_DIR / "humanqc_within_rater_duplicate_audit.csv")

if not within_rater_duplicates.empty:
    raise RuntimeError(
        "The same recording appears more than once for the same rater. "
        "Review humanqc_within_rater_duplicate_audit.csv."
    )


## 6. Parse JSON annotations into a raw interval ledger

Every interval remains linked to its source CSV row and 5-second interface segment. Parse failures, unknown item names, malformed intervals, and boundary violations are exported rather than silently discarded.


In [ ]:
interval_rows: list[dict[str, Any]] = []
parse_issue_rows: list[dict[str, Any]] = []
taxonomy_issue_rows: list[dict[str, Any]] = []

for row_index, row in raw_segments.iterrows():
    segment_onset = float(row["onset_seconds_absolute"])
    segment_offset = float(row["offset_seconds_absolute"])
    recording_duration = float(
        raw_segments.loc[
            raw_segments["recording_rating_id"] == row["recording_rating_id"],
            "offset_seconds_absolute",
        ].max()
    )

    for source_family in ANNOTATION_COLUMNS:
        parsed, parse_error = parse_json_dict(row[source_family])

        if parse_error is not None:
            parse_issue_rows.append({
                "annotation_csv_path": row["annotation_csv_path"],
                "row_index": row_index,
                "recording_id": row["recording_id"],
                "rater": row["rater"],
                "segment_number": row["segment_number"],
                "source_family": source_family,
                "parse_error": parse_error,
                "raw_value": str(row[source_family])[:1000],
            })
            continue

        expected_items = set(ANNOTATION_SCHEMA[source_family])
        observed_items = set(parsed)
        missing_items = sorted(expected_items - observed_items)
        unknown_items = sorted(observed_items - expected_items)

        if missing_items or unknown_items:
            taxonomy_issue_rows.append({
                "annotation_csv_path": row["annotation_csv_path"],
                "row_index": row_index,
                "recording_id": row["recording_id"],
                "rater": row["rater"],
                "segment_number": row["segment_number"],
                "source_family": source_family,
                "missing_items": " | ".join(missing_items),
                "unknown_items": " | ".join(unknown_items),
            })

        for item, intervals in parsed.items():
            if item not in expected_items:
                continue

            if intervals is None:
                intervals = []

            if not isinstance(intervals, list):
                parse_issue_rows.append({
                    "annotation_csv_path": row["annotation_csv_path"],
                    "row_index": row_index,
                    "recording_id": row["recording_id"],
                    "rater": row["rater"],
                    "segment_number": row["segment_number"],
                    "source_family": source_family,
                    "item": item,
                    "parse_error": f"Intervals must be a list, received {type(intervals).__name__}",
                    "raw_value": repr(intervals)[:1000],
                })
                continue

            for interval_index, interval in enumerate(intervals):
                if not (
                    isinstance(interval, (list, tuple))
                    and len(interval) == 2
                ):
                    parse_issue_rows.append({
                        "annotation_csv_path": row["annotation_csv_path"],
                        "row_index": row_index,
                        "recording_id": row["recording_id"],
                        "rater": row["rater"],
                        "segment_number": row["segment_number"],
                        "source_family": source_family,
                        "item": item,
                        "parse_error": "Interval must be a two-element sequence",
                        "raw_value": repr(interval)[:1000],
                    })
                    continue

                try:
                    raw_onset = float(interval[0])
                    raw_offset = float(interval[1])
                except Exception as exc:
                    parse_issue_rows.append({
                        "annotation_csv_path": row["annotation_csv_path"],
                        "row_index": row_index,
                        "recording_id": row["recording_id"],
                        "rater": row["rater"],
                        "segment_number": row["segment_number"],
                        "source_family": source_family,
                        "item": item,
                        "parse_error": f"Non-numeric interval: {type(exc).__name__}: {exc}",
                        "raw_value": repr(interval)[:1000],
                    })
                    continue

                finite = bool(np.isfinite(raw_onset) and np.isfinite(raw_offset))
                positive = bool(finite and raw_offset > raw_onset)
                within_recording = bool(
                    finite
                    and raw_onset >= -SEGMENT_BOUNDARY_TOLERANCE_SEC
                    and raw_offset <= recording_duration + SEGMENT_BOUNDARY_TOLERANCE_SEC
                )
                within_segment = bool(
                    finite
                    and raw_onset >= segment_onset - SEGMENT_BOUNDARY_TOLERANCE_SEC
                    and raw_offset <= segment_offset + SEGMENT_BOUNDARY_TOLERANCE_SEC
                )

                # Clip only negligible floating-point/GUI-boundary overflow.
                clipped_onset = (
                    max(0.0, raw_onset)
                    if raw_onset >= -SEGMENT_BOUNDARY_TOLERANCE_SEC
                    else raw_onset
                )
                clipped_offset = (
                    min(recording_duration, raw_offset)
                    if raw_offset <= recording_duration + SEGMENT_BOUNDARY_TOLERANCE_SEC
                    else raw_offset
                )

                interval_rows.append({
                    "recording_rating_id": row["recording_rating_id"],
                    "recording_id": row["recording_id"],
                    "rater": row["rater"],
                    "annotation_csv_path": row["annotation_csv_path"],
                    "segment_number": int(row["segment_number"]),
                    "segment_onset_sec": segment_onset,
                    "segment_offset_sec": segment_offset,
                    "recording_duration_sec": recording_duration,
                    "source_family": source_family,
                    "signal_family": SOURCE_TO_SIGNAL_FAMILY[source_family],
                    "item": item,
                    "interval_index_in_cell": interval_index,
                    "raw_onset_sec": raw_onset,
                    "raw_offset_sec": raw_offset,
                    "onset_sec": clipped_onset,
                    "offset_sec": clipped_offset,
                    "finite_interval": finite,
                    "positive_duration": positive,
                    "within_recording_tolerance": within_recording,
                    "within_segment_tolerance": within_segment,
                })

raw_interval_ledger = pd.DataFrame(interval_rows)
parse_issues = pd.DataFrame(parse_issue_rows)
taxonomy_issues = pd.DataFrame(taxonomy_issue_rows)

if not raw_interval_ledger.empty:
    raw_interval_ledger["valid_interval"] = (
        raw_interval_ledger["finite_interval"]
        & raw_interval_ledger["positive_duration"]
        & raw_interval_ledger["within_recording_tolerance"]
    )
    raw_interval_ledger["duration_sec"] = (
        raw_interval_ledger["offset_sec"] - raw_interval_ledger["onset_sec"]
    )
    raw_interval_ledger["exact_duplicate"] = raw_interval_ledger.duplicated(
        subset=[
            "recording_rating_id",
            "source_family",
            "item",
            "onset_sec",
            "offset_sec",
        ],
        keep="first",
    )
else:
    raw_interval_ledger["valid_interval"] = pd.Series(dtype=bool)
    raw_interval_ledger["duration_sec"] = pd.Series(dtype=float)
    raw_interval_ledger["exact_duplicate"] = pd.Series(dtype=bool)

display(raw_interval_ledger.head())
display(parse_issues.head())
display(taxonomy_issues.head())

export_dataframe(raw_interval_ledger, TABLE_DIR / "humanqc_raw_interval_ledger.csv")
export_dataframe(parse_issues, LOG_DIR / "humanqc_json_parse_issues.csv")
export_dataframe(taxonomy_issues, LOG_DIR / "humanqc_taxonomy_issues.csv")

print("Raw intervals:", len(raw_interval_ledger))
print("Valid intervals:", int(raw_interval_ledger["valid_interval"].sum()))
print("Exact duplicate intervals:", int(raw_interval_ledger["exact_duplicate"].sum()))
print("Parse issues:", len(parse_issues))
print("Taxonomy issues:", len(taxonomy_issues))


## 7. Adjacency-gap and merge-tolerance sensitivity

The annotation interface divides recordings into approximately 5-second rows. A single continuous percept may therefore appear as adjacent intervals in consecutive rows. The notebook compares 0, 1, 10, 50, and 100 ms union tolerances before using the **provisional 10 ms** Stage-1 tolerance. This tolerance is not considered frozen until the audit is reviewed.


In [ ]:
valid_intervals = raw_interval_ledger.loc[
    raw_interval_ledger["valid_interval"]
    & ~raw_interval_ledger["exact_duplicate"]
].copy()

gap_rows = []
for keys, group in valid_intervals.groupby(
    ["recording_rating_id", "source_family", "item"],
    sort=False,
):
    ordered = group.sort_values(["onset_sec", "offset_sec"])
    values = ordered[["onset_sec", "offset_sec"]].to_numpy(float)
    for previous, current in zip(values[:-1], values[1:]):
        gap_rows.append({
            "recording_rating_id": keys[0],
            "source_family": keys[1],
            "item": keys[2],
            "previous_offset_sec": previous[1],
            "next_onset_sec": current[0],
            "gap_sec": current[0] - previous[1],
        })

adjacency_gaps = pd.DataFrame(gap_rows)
positive_gaps = adjacency_gaps.loc[adjacency_gaps["gap_sec"] >= 0].copy()

gap_threshold_summary = pd.DataFrame([
    {
        "threshold_sec": tolerance,
        "adjacent_pairs_at_or_below": int((positive_gaps["gap_sec"] <= tolerance).sum())
        if not positive_gaps.empty else 0,
        "percent_of_nonnegative_gaps": (
            100 * float((positive_gaps["gap_sec"] <= tolerance).mean())
            if not positive_gaps.empty else np.nan
        ),
    }
    for tolerance in MERGE_SENSITIVITY_TOLERANCES_SEC
])

sensitivity_rows = []
group_columns = ["recording_rating_id", "source_family", "item"]

for tolerance in MERGE_SENSITIVITY_TOLERANCES_SEC:
    for keys, group in valid_intervals.groupby(group_columns, sort=False):
        intervals = list(zip(group["onset_sec"], group["offset_sec"]))
        merged = merge_intervals(intervals, tolerance)
        summary = summarize_intervals(merged)
        sensitivity_rows.append({
            "recording_rating_id": keys[0],
            "source_family": keys[1],
            "item": keys[2],
            "merge_tolerance_sec": tolerance,
            **summary,
        })

merge_sensitivity_detail = pd.DataFrame(sensitivity_rows)
merge_sensitivity_summary = (
    merge_sensitivity_detail.groupby("merge_tolerance_sec")
    .agg(
        total_merged_events=("event_count", "sum"),
        total_union_duration_sec=("union_duration_sec", "sum"),
        recording_item_combinations=("recording_rating_id", "size"),
    )
    .reset_index()
)

display(gap_threshold_summary)
display(merge_sensitivity_summary)

export_dataframe(adjacency_gaps, LOG_DIR / "humanqc_interval_adjacency_gaps.csv")
export_dataframe(gap_threshold_summary, TABLE_DIR / "humanqc_gap_threshold_summary.csv")
export_dataframe(merge_sensitivity_detail, TABLE_DIR / "humanqc_merge_sensitivity_detail.csv")
export_dataframe(merge_sensitivity_summary, TABLE_DIR / "humanqc_merge_sensitivity_summary.csv")


## 8. Build the provisional merged interval ledger

In [ ]:
merged_rows = []

for keys, group in valid_intervals.groupby(
    ["recording_rating_id", "recording_id", "rater", "source_family", "signal_family", "item"],
    sort=False,
):
    intervals = list(zip(group["onset_sec"], group["offset_sec"]))
    merged = merge_intervals(intervals, PROVISIONAL_MERGE_TOLERANCE_SEC)
    recording_duration = float(group["recording_duration_sec"].max())

    for event_index, (onset, offset) in enumerate(merged, start=1):
        merged_rows.append({
            "recording_rating_id": keys[0],
            "recording_id": keys[1],
            "rater": keys[2],
            "source_family": keys[3],
            "signal_family": keys[4],
            "item": keys[5],
            "event_index": event_index,
            "onset_sec": onset,
            "offset_sec": offset,
            "duration_sec": offset - onset,
            "recording_duration_sec": recording_duration,
            "merge_tolerance_sec": PROVISIONAL_MERGE_TOLERANCE_SEC,
        })

merged_interval_ledger = pd.DataFrame(merged_rows)

display(merged_interval_ledger.head())
export_dataframe(merged_interval_ledger, TABLE_DIR / "humanqc_merged_interval_ledger_PROVISIONAL.csv")


## 9. Recording–item, source-family, and six-family summaries

Summary duration is calculated from the union of intervals, not by summing overlapping item durations. QADD is reconstructed by unioning `Environmental noise` and `Competing speech`. The task-adherence family remains separate.


In [ ]:
rating_base = (
    raw_segments.groupby(["recording_rating_id", "recording_id", "rater"], as_index=False)
    .agg(recording_duration_sec=("offset_seconds_absolute", "max"))
)

# Complete item grid ensures valid zeros are explicit.
item_grid = (
    rating_base.assign(_key=1)
    .merge(annotation_taxonomy.assign(_key=1), on="_key", how="outer")
    .drop(columns="_key")
)

item_event_summary = (
    merged_interval_ledger.groupby(
        ["recording_rating_id", "recording_id", "rater", "source_family", "signal_family", "item"],
        as_index=False,
    )
    .agg(
        event_count=("event_index", "max"),
        union_duration_sec=("duration_sec", "sum"),
        first_onset_sec=("onset_sec", "min"),
        last_offset_sec=("offset_sec", "max"),
    )
)

recording_item_summary = item_grid.merge(
    item_event_summary,
    on=["recording_rating_id", "recording_id", "rater", "source_family", "signal_family", "item"],
    how="left",
)

recording_item_summary["event_count"] = recording_item_summary["event_count"].fillna(0).astype(int)
recording_item_summary["union_duration_sec"] = recording_item_summary["union_duration_sec"].fillna(0.0)
recording_item_summary["present"] = recording_item_summary["event_count"] > 0
recording_item_summary["duration_fraction"] = np.where(
    recording_item_summary["recording_duration_sec"] > 0,
    recording_item_summary["union_duration_sec"] / recording_item_summary["recording_duration_sec"],
    np.nan,
)

# Source-family union: merge all item intervals within the GUI family.
source_family_rows = []
for keys, group in merged_interval_ledger.groupby(
    ["recording_rating_id", "recording_id", "rater", "source_family", "signal_family"],
    sort=False,
):
    merged = merge_intervals(
        list(zip(group["onset_sec"], group["offset_sec"])),
        PROVISIONAL_MERGE_TOLERANCE_SEC,
    )
    summary = summarize_intervals(merged)
    source_family_rows.append({
        "recording_rating_id": keys[0],
        "recording_id": keys[1],
        "rater": keys[2],
        "source_family": keys[3],
        "signal_family": keys[4],
        "recording_duration_sec": float(group["recording_duration_sec"].max()),
        "distinct_items_present": int(group["item"].nunique()),
        **summary,
    })

source_family_events = pd.DataFrame(source_family_rows)

source_family_grid = (
    rating_base.assign(_key=1)
    .merge(
        pd.DataFrame({
            "source_family": ANNOTATION_COLUMNS,
            "signal_family": [SOURCE_TO_SIGNAL_FAMILY[value] for value in ANNOTATION_COLUMNS],
        }).assign(_key=1),
        on="_key",
        how="outer",
    )
    .drop(columns="_key")
)

recording_source_family_summary = source_family_grid.merge(
    source_family_events,
    on=[
        "recording_rating_id",
        "recording_id",
        "rater",
        "source_family",
        "signal_family",
        "recording_duration_sec",
    ],
    how="left",
)

for column, default in [
    ("distinct_items_present", 0),
    ("event_count", 0),
    ("union_duration_sec", 0.0),
]:
    recording_source_family_summary[column] = (
        recording_source_family_summary[column].fillna(default)
    )

recording_source_family_summary["distinct_items_present"] = (
    recording_source_family_summary["distinct_items_present"].astype(int)
)
recording_source_family_summary["event_count"] = (
    recording_source_family_summary["event_count"].astype(int)
)
recording_source_family_summary["present"] = (
    recording_source_family_summary["event_count"] > 0
)
recording_source_family_summary["duration_fraction"] = np.where(
    recording_source_family_summary["recording_duration_sec"] > 0,
    recording_source_family_summary["union_duration_sec"]
    / recording_source_family_summary["recording_duration_sec"],
    np.nan,
)

# Six-family union, including QADD union across environmental noise and competing speech.
signal_family_rows = []
for keys, group in merged_interval_ledger.groupby(
    ["recording_rating_id", "recording_id", "rater", "signal_family"],
    sort=False,
):
    merged = merge_intervals(
        list(zip(group["onset_sec"], group["offset_sec"])),
        PROVISIONAL_MERGE_TOLERANCE_SEC,
    )
    summary = summarize_intervals(merged)
    signal_family_rows.append({
        "recording_rating_id": keys[0],
        "recording_id": keys[1],
        "rater": keys[2],
        "signal_family": keys[3],
        "recording_duration_sec": float(group["recording_duration_sec"].max()),
        "distinct_source_families_present": int(group["source_family"].nunique()),
        "distinct_items_present": int(group["item"].nunique()),
        **summary,
    })

signal_family_events = pd.DataFrame(signal_family_rows)
signal_family_values = ["QADD", "QGAIN", "QREV", "QCHAN", "QDIST", "QTEMP", "TASK"]

signal_family_grid = (
    rating_base.assign(_key=1)
    .merge(
        pd.DataFrame({"signal_family": signal_family_values}).assign(_key=1),
        on="_key",
        how="outer",
    )
    .drop(columns="_key")
)

recording_signal_family_summary = signal_family_grid.merge(
    signal_family_events,
    on=[
        "recording_rating_id",
        "recording_id",
        "rater",
        "signal_family",
        "recording_duration_sec",
    ],
    how="left",
)

for column, default in [
    ("distinct_source_families_present", 0),
    ("distinct_items_present", 0),
    ("event_count", 0),
    ("union_duration_sec", 0.0),
]:
    recording_signal_family_summary[column] = (
        recording_signal_family_summary[column].fillna(default)
    )

for column in ["distinct_source_families_present", "distinct_items_present", "event_count"]:
    recording_signal_family_summary[column] = recording_signal_family_summary[column].astype(int)

recording_signal_family_summary["present"] = (
    recording_signal_family_summary["event_count"] > 0
)
recording_signal_family_summary["duration_fraction"] = np.where(
    recording_signal_family_summary["recording_duration_sec"] > 0,
    recording_signal_family_summary["union_duration_sec"]
    / recording_signal_family_summary["recording_duration_sec"],
    np.nan,
)

display(recording_item_summary.head())
display(recording_signal_family_summary.head())

export_dataframe(recording_item_summary, TABLE_DIR / "humanqc_recording_item_summary.csv")
export_dataframe(
    recording_source_family_summary,
    TABLE_DIR / "humanqc_recording_source_family_summary.csv",
)
export_dataframe(
    recording_signal_family_summary,
    TABLE_DIR / "humanqc_recording_signal_family_summary.csv",
)


## 10. Discover the frozen 519-recording population and metadata

In [ ]:
# ---------------------------------------------------------------------
# Reconstruct the frozen 519-recording population and attach metadata
# ---------------------------------------------------------------------

def discover_id_tables(
    root: Path,
) -> tuple[pd.DataFrame, dict[str, set[str]]]:
    candidate_rows = []
    candidate_sets: dict[str, set[str]] = {}

    if not root.exists():
        return pd.DataFrame(), candidate_sets

    paths = sorted(
        list(root.rglob("*.csv"))
        + list(root.rglob("*.parquet"))
        + list(root.rglob("*.pq"))
    )

    for path in paths:
        try:
            if path.suffix.casefold() == ".csv":
                header = pd.read_csv(path, nrows=0)

            else:
                header = pd.read_parquet(path).head(0)

        except Exception:
            continue

        lower_to_original = {
            str(column).casefold(): column
            for column in header.columns
        }

        id_column = next(
            (
                lower_to_original[candidate.casefold()]
                for candidate in ID_COLUMN_CANDIDATES
                if candidate.casefold() in lower_to_original
            ),
            None,
        )

        if id_column is None:
            continue

        try:
            ids = (
                read_tabular(
                    path,
                    usecols=[id_column],
                )[id_column]
                .map(canonical_recording_id)
            )

        except Exception:
            continue

        id_set = set(ids) - {""}
        key = str(path)

        candidate_sets[key] = id_set

        candidate_rows.append({
            "path": key,
            "id_column": id_column,
            "rows": len(ids),
            "unique_ids": len(id_set),
            "exact_519_unique": (
                len(id_set) == FINAL_POPULATION_N
            ),
        })

    return pd.DataFrame(candidate_rows), candidate_sets


# ---------------------------------------------------------------------
# Frozen feature population
# ---------------------------------------------------------------------

population_candidate_table, population_candidate_sets = (
    discover_id_tables(FAMILY_FREEZE_ROOT)
)

if population_candidate_table.empty:
    raise RuntimeError(
        "No tables containing recognizable recording IDs were found "
        f"under {FAMILY_FREEZE_ROOT}."
    )

display(
    population_candidate_table.sort_values(
        [
            "exact_519_unique",
            "unique_ids",
        ],
        ascending=[False, False],
    ).head(30)
)

export_dataframe(
    population_candidate_table,
    LOG_DIR / "frozen_population_candidate_tables.csv",
)

exact_population_sets = [
    id_set
    for id_set in population_candidate_sets.values()
    if len(id_set) == FINAL_POPULATION_N
]

if not exact_population_sets:
    raise RuntimeError(
        "Could not find a frozen family table with exactly "
        f"{FINAL_POPULATION_N} unique recording IDs."
    )

population_intersection = set.intersection(
    *exact_population_sets
)

population_union = set.union(
    *exact_population_sets
)

if len(population_intersection) != FINAL_POPULATION_N:
    raise RuntimeError(
        "Tables with 519 unique IDs do not agree on one common "
        "519-recording population. "
        f"Intersection={len(population_intersection)}, "
        f"union={len(population_union)}."
    )

frozen_population = pd.DataFrame({
    "recording_id": sorted(population_intersection)
})

print(
    "Frozen feature population reconstructed:",
    len(frozen_population),
    "recordings",
)


# ---------------------------------------------------------------------
# Flexible metadata discovery
# ---------------------------------------------------------------------

def normalize_column_name(value: Any) -> str:
    return re.sub(
        r"[^a-z0-9]+",
        "",
        str(value).strip().casefold(),
    )


def find_metadata_column(
    columns: Iterable[Any],
    aliases: list[str],
    keyword_groups: list[tuple[str, ...]] | None = None,
) -> str | None:
    """
    Find a metadata column using exact normalized aliases first,
    followed by conservative keyword matching.
    """
    keyword_groups = keyword_groups or []

    normalized_to_original = {
        normalize_column_name(column): str(column)
        for column in columns
    }

    for alias in aliases:
        normalized_alias = normalize_column_name(alias)

        if normalized_alias in normalized_to_original:
            return normalized_to_original[normalized_alias]

    for keyword_group in keyword_groups:
        matches = []

        for normalized, original in normalized_to_original.items():
            if all(
                normalize_column_name(keyword) in normalized
                for keyword in keyword_group
            ):
                matches.append(
                    (
                        len(normalized),
                        original,
                    )
                )

        if matches:
            return sorted(matches)[0][1]

    return None


def read_metadata_table(
    path: Path,
    usecols: list[str] | None = None,
) -> pd.DataFrame:
    suffix = path.suffix.casefold()

    if suffix == ".csv":
        return pd.read_csv(
            path,
            usecols=usecols,
            low_memory=False,
        )

    if suffix in {".parquet", ".pq"}:
        frame = pd.read_parquet(path)

        if usecols is not None:
            frame = frame[usecols]

        return frame

    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(
            path,
            usecols=usecols,
        )

    raise ValueError(
        f"Unsupported metadata table type: {path}"
    )


def canonical_participant_id(value: Any) -> str:
    if value is None or pd.isna(value):
        return ""

    if isinstance(value, (int, np.integer)):
        return str(int(value))

    if isinstance(value, (float, np.floating)):
        if np.isfinite(value) and float(value).is_integer():
            return str(int(value))

    text = str(value).strip()

    if text.casefold() in {
        "",
        "nan",
        "none",
        "null",
        "na",
    }:
        return ""

    return re.sub(
        r"\s+",
        "",
        text,
    ).casefold()


def normalize_cohort(
    value: Any,
    source_column: str = "",
) -> str | float:
    if value is None or pd.isna(value):
        return np.nan

    source_name = normalize_column_name(source_column)
    raw_text = str(value).strip()
    text = raw_text.casefold()

    if text in {
        "",
        "nan",
        "none",
        "null",
        "na",
        "unknown",
        "missing",
    }:
        return np.nan

    control_values = {
        "control",
        "controls",
        "ctrl",
        "hc",
        "healthycontrol",
        "healthy control",
        "controlparticipant",
        "control participant",
        "nonals",
        "non-als",
        "healthy",
        "unaffected",
    }

    als_values = {
        "als",
        "amyotrophiclateralsclerosis",
        "amyotrophic lateral sclerosis",
        "alsparticipant",
        "als participant",
        "patient",
        "case",
    }

    compact = normalize_column_name(raw_text)

    if text in control_values or compact in {
        normalize_column_name(item)
        for item in control_values
    }:
        return "CONTROL"

    if text in als_values or compact in {
        normalize_column_name(item)
        for item in als_values
    }:
        return "ALS"

    if (
        "control" in text
        or "healthy" in text
        or "unaffected" in text
    ):
        return "CONTROL"

    if (
        re.search(r"\bals\b", text)
        or "amyotrophic" in text
    ):
        return "ALS"

    # Interpret binary values only when the column itself clearly
    # represents ALS/control status.
    binary_status_column = any(
        token in source_name
        for token in [
            "isals",
            "alsflag",
            "alscontrol",
            "casecontrol",
            "cohortcode",
        ]
    )

    if binary_status_column:
        if text in {"1", "1.0", "true", "yes", "y"}:
            return "ALS"

        if text in {"0", "0.0", "false", "no", "n"}:
            return "CONTROL"

    return np.nan


RECORDING_ID_ALIASES = [
    *ID_COLUMN_CANDIDATES,
    "recording",
    "recording_name",
    "recording_file",
    "recording_path",
    "audio_path",
    "audio_name",
    "file",
    "fname",
    "task_filename",
    "original_filename",
    "source_file",
]

PARTICIPANT_ID_ALIASES = [
    *PARTICIPANT_COLUMN_CANDIDATES,
    "participant_uid",
    "participant_code",
    "participant_number",
    "subject_uid",
    "subject_code",
    "subject_number",
    "patient_id",
    "patient_uid",
    "patient_code",
    "speaker_id",
    "speaker_uid",
    "speaker",
    "person_id",
    "research_id",
    "study_id",
    "uid",
    "subjid",
    "subjectid",
    "participantid",
]

COHORT_ALIASES = [
    *COHORT_COLUMN_CANDIDATES,
    "cohort_label",
    "cohort_name",
    "group_label",
    "group_name",
    "diagnosis_label",
    "diagnosis_name",
    "diagnostic_category",
    "clinical_group",
    "clinical_status",
    "participant_group",
    "participant_type",
    "subject_group",
    "subject_type",
    "population",
    "disease_group",
    "disease_status",
    "dx",
    "dx_group",
    "case_control",
    "casecontrol",
    "als_control",
    "is_als",
    "als_flag",
]


# ---------------------------------------------------------------------
# Catalogue all metadata tables and their possible columns
# ---------------------------------------------------------------------

if not METADATA_ROOT.exists():
    raise FileNotFoundError(
        f"Metadata root does not exist: {METADATA_ROOT}"
    )

metadata_paths = sorted(
    list(METADATA_ROOT.rglob("*.csv"))
    + list(METADATA_ROOT.rglob("*.parquet"))
    + list(METADATA_ROOT.rglob("*.pq"))
    + list(METADATA_ROOT.rglob("*.xlsx"))
    + list(METADATA_ROOT.rglob("*.xls"))
)

metadata_catalog_rows = []

for path in metadata_paths:
    try:
        suffix = path.suffix.casefold()

        if suffix == ".csv":
            header = pd.read_csv(path, nrows=0)

        elif suffix in {".parquet", ".pq"}:
            header = pd.read_parquet(path).head(0)

        else:
            header = pd.read_excel(
                path,
                nrows=0,
            )

    except Exception as exc:
        metadata_catalog_rows.append({
            "path": str(path),
            "readable": False,
            "read_error": (
                f"{type(exc).__name__}: {exc}"
            ),
            "id_column": None,
            "participant_column": None,
            "cohort_column": None,
        })
        continue

    id_column = find_metadata_column(
        header.columns,
        RECORDING_ID_ALIASES,
        keyword_groups=[
            ("recording", "id"),
            ("recording", "file"),
            ("audio", "file"),
            ("file", "name"),
        ],
    )

    participant_column = find_metadata_column(
        header.columns,
        PARTICIPANT_ID_ALIASES,
        keyword_groups=[
            ("participant", "id"),
            ("subject", "id"),
            ("patient", "id"),
            ("speaker", "id"),
        ],
    )

    cohort_column = find_metadata_column(
        header.columns,
        COHORT_ALIASES,
        keyword_groups=[
            ("cohort",),
            ("diagnosis",),
            ("clinical", "group"),
            ("disease", "group"),
            ("case", "control"),
            ("als", "control"),
        ],
    )

    metadata_catalog_rows.append({
        "path": str(path),
        "readable": True,
        "read_error": None,
        "rows": np.nan,
        "columns": len(header.columns),
        "id_column": id_column,
        "participant_column": participant_column,
        "cohort_column": cohort_column,
        "column_names": " | ".join(
            map(str, header.columns)
        ),
    })

metadata_catalog = pd.DataFrame(
    metadata_catalog_rows
)

export_dataframe(
    metadata_catalog,
    LOG_DIR / "metadata_table_catalog.csv",
)

display(
    metadata_catalog.loc[
        metadata_catalog["readable"]
    ][
        [
            "path",
            "id_column",
            "participant_column",
            "cohort_column",
        ]
    ]
)


# ---------------------------------------------------------------------
# Discover the recording-to-participant map
# ---------------------------------------------------------------------

recording_participant_candidate_rows = []
recording_participant_frames = {}

for row in metadata_catalog.itertuples(index=False):
    if (
        not bool(row.readable)
        or not row.id_column
        or not row.participant_column
    ):
        continue

    path = Path(row.path)

    try:
        frame = read_metadata_table(
            path,
            usecols=[
                row.id_column,
                row.participant_column,
            ],
        )

    except Exception as exc:
        recording_participant_candidate_rows.append({
            "path": str(path),
            "id_column": row.id_column,
            "participant_column": row.participant_column,
            "load_error": (
                f"{type(exc).__name__}: {exc}"
            ),
            "population_overlap_unique": 0,
            "participant_nonmissing": 0,
        })
        continue

    candidate = pd.DataFrame({
        "recording_id": (
            frame[row.id_column]
            .map(canonical_recording_id)
        ),
        "participant_id": (
            frame[row.participant_column]
            .map(canonical_participant_id)
        ),
    })

    candidate = candidate.loc[
        candidate["recording_id"].isin(
            population_intersection
        )
    ]

    overlap_unique = (
        candidate["recording_id"].nunique()
    )

    participant_nonmissing = (
        candidate.loc[
            candidate["participant_id"].ne(""),
            "recording_id",
        ].nunique()
    )

    recording_participant_candidate_rows.append({
        "path": str(path),
        "id_column": row.id_column,
        "participant_column": row.participant_column,
        "load_error": None,
        "rows": len(frame),
        "population_overlap_unique": overlap_unique,
        "participant_nonmissing": participant_nonmissing,
    })

    recording_participant_frames[str(path)] = candidate

recording_participant_candidates = pd.DataFrame(
    recording_participant_candidate_rows
)

export_dataframe(
    recording_participant_candidates,
    LOG_DIR / "recording_participant_candidates.csv",
)

if recording_participant_candidates.empty:
    raise RuntimeError(
        "No table containing both recording ID and participant ID "
        "could be identified. Review metadata_table_catalog.csv."
    )

valid_participant_candidates = (
    recording_participant_candidates.loc[
        recording_participant_candidates[
            "load_error"
        ].isna()
    ]
    .sort_values(
        [
            "participant_nonmissing",
            "population_overlap_unique",
            "rows",
        ],
        ascending=[False, False, False],
    )
)

best_participant_source = (
    valid_participant_candidates.iloc[0]
)

participant_source_path = str(
    best_participant_source["path"]
)

participant_map_raw = (
    recording_participant_frames[
        participant_source_path
    ].copy()
)

participant_map_raw = participant_map_raw.loc[
    participant_map_raw["recording_id"].isin(
        population_intersection
    )
    & participant_map_raw["participant_id"].ne("")
]

participant_conflicts = (
    participant_map_raw.groupby(
        "recording_id"
    )["participant_id"]
    .nunique()
    .reset_index(name="participant_ids")
)

participant_conflicts = (
    participant_conflicts.loc[
        participant_conflicts[
            "participant_ids"
        ] > 1
    ]
)

export_dataframe(
    participant_conflicts,
    LOG_DIR / "metadata_participant_conflicts.csv",
)

if not participant_conflicts.empty:
    display(participant_conflicts)

    raise RuntimeError(
        "At least one recording maps to multiple participant IDs. "
        "Review metadata_participant_conflicts.csv."
    )

participant_map = (
    participant_map_raw[
        [
            "recording_id",
            "participant_id",
        ]
    ]
    .drop_duplicates(
        subset=["recording_id"],
        keep="first",
    )
)

print(
    "Selected recording-participant source:",
    participant_source_path,
)

print(
    "Participant coverage:",
    participant_map["recording_id"].nunique(),
    "/",
    FINAL_POPULATION_N,
)


# ---------------------------------------------------------------------
# Discover cohort: direct recording-level candidates
# ---------------------------------------------------------------------

direct_cohort_candidate_rows = []
direct_cohort_frames = {}

for row in metadata_catalog.itertuples(index=False):
    if (
        not bool(row.readable)
        or not row.id_column
        or not row.cohort_column
    ):
        continue

    path = Path(row.path)

    try:
        frame = read_metadata_table(
            path,
            usecols=[
                row.id_column,
                row.cohort_column,
            ],
        )

    except Exception as exc:
        continue

    candidate = pd.DataFrame({
        "recording_id": (
            frame[row.id_column]
            .map(canonical_recording_id)
        ),
        "cohort": [
            normalize_cohort(
                value,
                source_column=row.cohort_column,
            )
            for value in frame[
                row.cohort_column
            ]
        ],
    })

    candidate = candidate.loc[
        candidate["recording_id"].isin(
            population_intersection
        )
    ]

    coverage = candidate.loc[
        candidate["cohort"].notna(),
        "recording_id",
    ].nunique()

    direct_cohort_candidate_rows.append({
        "source_type": "recording_level",
        "path": str(path),
        "id_column": row.id_column,
        "participant_column": None,
        "cohort_column": row.cohort_column,
        "population_coverage": coverage,
    })

    direct_cohort_frames[str(path)] = candidate


# ---------------------------------------------------------------------
# Resolve cohort from all valid metadata sources
# ---------------------------------------------------------------------

EXPECTED_COHORT_RECORDINGS = {
    "ALS": 418,
    "CONTROL": 101,
}
EXPECTED_PARTICIPANTS = 224


def normalize_cohort_strict(
    value: Any,
    source_column: str = "",
) -> str | float:
    """
    Normalize explicit diagnostic/cohort labels.

    Eligibility flags are never interpreted as cohort labels.
    Generic labels such as 'patient' are deliberately not mapped.
    """
    if value is None or pd.isna(value):
        return np.nan

    raw = str(value).strip()

    if raw.casefold() in {
        "",
        "nan",
        "none",
        "null",
        "na",
        "n/a",
        "unknown",
        "missing",
    }:
        return np.nan

    text = raw.casefold()
    compact = re.sub(
        r"[^a-z0-9]+",
        "",
        text,
    )

    control_exact = {
        "control",
        "controls",
        "ctrl",
        "hc",
        "healthy",
        "healthycontrol",
        "controlparticipant",
        "nonals",
        "unaffected",
    }

    als_exact = {
        "als",
        "alsparticipant",
        "amyotrophiclateralsclerosis",
        "mnd",
        "motorneurondisease",
        "alsmnd",
    }

    source_compact = re.sub(
        r"[^a-z0-9]+",
        "",
        str(source_column).casefold(),
    )

    # The governed disposition table explicitly confirms control status for
    # records whose original Diagnosis value is missing. Do not interpret
    # generic disposition values (for example, reported_metadata) as cohorts.
    if (
        source_compact == "diagnosisresolution"
        and compact in {
            "confirmedcontrolidrule",
            "investigatorconfirmedcontrolsubject",
        }
    ):
        return "CONTROL"

    if compact in control_exact:
        return "CONTROL"

    if compact in als_exact:
        return "ALS"

    if any(
        phrase in text
        for phrase in [
            "healthy control",
            "control participant",
            "non-als",
            "non als",
            "unaffected",
        ]
    ):
        return "CONTROL"

    if (
        re.search(r"\bals\b", text)
        or "amyotrophic lateral sclerosis" in text
        or "motor neuron disease" in text
    ):
        return "ALS"

    return np.nan


def cohort_column_priority(
    column_name: str,
) -> int | None:
    """
    Lower values indicate more authoritative cohort fields.

    Eligibility fields are excluded because they are not cohort labels.
    """
    normalized = re.sub(
        r"[^a-z0-9]+",
        "",
        str(column_name).casefold(),
    )

    excluded_tokens = [
        "eligible",
        "eligibility",
        "included",
        "excluded",
        "inclusion",
        "exclusion",
    ]

    if any(token in normalized for token in excluded_tokens):
        return None

    if "diagnosisresolution" in normalized:
        return 0

    if "diagnosisanalysis" in normalized:
        return 1

    if normalized in {
        "cohort",
        "cohortlabel",
        "cohortname",
        "group",
        "diagnosticgroup",
        "clinicalgroup",
    }:
        return 2

    if "diagnosis" in normalized:
        return 3

    if "cohort" in normalized:
        return 4

    if "group" in normalized:
        return 5

    return 10


# ---------------------------------------------------------------------
# Collect every usable recording-level and participant-level label
# ---------------------------------------------------------------------

cohort_candidate_rows = []
cohort_source_audit_rows = []

for catalog_row in metadata_catalog.itertuples(index=False):
    if not bool(catalog_row.readable):
        continue

    cohort_column = catalog_row.cohort_column

    if not cohort_column:
        continue

    priority = cohort_column_priority(
        cohort_column
    )

    if priority is None:
        cohort_source_audit_rows.append({
            "path": catalog_row.path,
            "source_type": "excluded",
            "cohort_column": cohort_column,
            "priority": np.nan,
            "rows": np.nan,
            "normalized_labels": 0,
            "population_coverage": 0,
            "raw_values": "",
            "reason": (
                "Eligibility/inclusion field; "
                "not a cohort label"
            ),
        })
        continue

    path = Path(catalog_row.path)

    # -------------------------------------------------------------
    # Recording-level source
    # -------------------------------------------------------------
    if catalog_row.id_column:
        try:
            frame = read_metadata_table(
                path,
                usecols=[
                    catalog_row.id_column,
                    cohort_column,
                ],
            )
        except Exception as exc:
            cohort_source_audit_rows.append({
                "path": str(path),
                "source_type": "recording_level",
                "cohort_column": cohort_column,
                "priority": priority,
                "rows": np.nan,
                "normalized_labels": 0,
                "population_coverage": 0,
                "raw_values": "",
                "reason": (
                    f"{type(exc).__name__}: {exc}"
                ),
            })
        else:
            candidate = pd.DataFrame({
                "recording_id": (
                    frame[catalog_row.id_column]
                    .map(canonical_recording_id)
                ),
                "raw_cohort_value": (
                    frame[cohort_column]
                ),
            })

            candidate = candidate.loc[
                candidate["recording_id"].isin(
                    population_intersection
                )
            ].copy()

            candidate["cohort"] = [
                normalize_cohort_strict(
                    value,
                    source_column=cohort_column,
                )
                for value in candidate[
                    "raw_cohort_value"
                ]
            ]

            normalized = candidate.loc[
                candidate["cohort"].notna()
            ].copy()

            for row in normalized.itertuples(
                index=False
            ):
                cohort_candidate_rows.append({
                    "recording_id": (
                        row.recording_id
                    ),
                    "cohort": row.cohort,
                    "raw_cohort_value": (
                        row.raw_cohort_value
                    ),
                    "source_path": str(path),
                    "source_type": (
                        "recording_level"
                    ),
                    "cohort_column": (
                        cohort_column
                    ),
                    "priority": priority,
                })

            raw_values = sorted(
                {
                    str(value)
                    for value in candidate[
                        "raw_cohort_value"
                    ].dropna().unique()
                }
            )

            cohort_source_audit_rows.append({
                "path": str(path),
                "source_type": (
                    "recording_level"
                ),
                "cohort_column": (
                    cohort_column
                ),
                "priority": priority,
                "rows": len(frame),
                "normalized_labels": (
                    len(normalized)
                ),
                "population_coverage": (
                    normalized[
                        "recording_id"
                    ].nunique()
                ),
                "raw_values": " | ".join(
                    raw_values
                ),
                "reason": "",
            })

    # -------------------------------------------------------------
    # Participant-level source
    # -------------------------------------------------------------
    if catalog_row.participant_column:
        try:
            frame = read_metadata_table(
                path,
                usecols=[
                    catalog_row.participant_column,
                    cohort_column,
                ],
            )
        except Exception as exc:
            cohort_source_audit_rows.append({
                "path": str(path),
                "source_type": (
                    "participant_level"
                ),
                "cohort_column": (
                    cohort_column
                ),
                "priority": priority + 20,
                "rows": np.nan,
                "normalized_labels": 0,
                "population_coverage": 0,
                "raw_values": "",
                "reason": (
                    f"{type(exc).__name__}: {exc}"
                ),
            })
        else:
            participant_candidate = pd.DataFrame({
                "participant_id": (
                    frame[
                        catalog_row.participant_column
                    ].map(
                        canonical_participant_id
                    )
                ),
                "raw_cohort_value": (
                    frame[cohort_column]
                ),
            })

            participant_candidate["cohort"] = [
                normalize_cohort_strict(
                    value,
                    source_column=cohort_column,
                )
                for value in participant_candidate[
                    "raw_cohort_value"
                ]
            ]

            participant_candidate = (
                participant_candidate.loc[
                    participant_candidate[
                        "participant_id"
                    ].ne("")
                    & participant_candidate[
                        "cohort"
                    ].notna()
                ]
                .copy()
            )

            participant_conflicts = (
                participant_candidate.groupby(
                    "participant_id"
                )["cohort"]
                .nunique()
            )

            participant_conflicts = (
                participant_conflicts.loc[
                    participant_conflicts > 1
                ]
            )

            if not participant_conflicts.empty:
                conflict_detail = (
                    participant_candidate.loc[
                        participant_candidate[
                            "participant_id"
                        ].isin(
                            participant_conflicts.index
                        )
                    ]
                    .sort_values(
                        [
                            "participant_id",
                            "cohort",
                        ]
                    )
                )

                export_dataframe(
                    conflict_detail,
                    LOG_DIR
                    / (
                        "cohort_conflicts_"
                        + re.sub(
                            r"[^A-Za-z0-9]+",
                            "_",
                            path.stem,
                        )
                        + ".csv"
                    ),
                )

                cohort_source_audit_rows.append({
                    "path": str(path),
                    "source_type": (
                        "participant_level"
                    ),
                    "cohort_column": (
                        cohort_column
                    ),
                    "priority": priority + 20,
                    "rows": len(frame),
                    "normalized_labels": (
                        len(participant_candidate)
                    ),
                    "population_coverage": 0,
                    "raw_values": "",
                    "reason": (
                        "Participant-level cohort "
                        "conflicts detected"
                    ),
                })

            else:
                participant_candidate = (
                    participant_candidate
                    .drop_duplicates(
                        subset=[
                            "participant_id"
                        ],
                        keep="first",
                    )
                )

                recording_candidate = (
                    participant_map.merge(
                        participant_candidate,
                        on="participant_id",
                        how="left",
                        validate="many_to_one",
                    )
                    .dropna(
                        subset=["cohort"]
                    )
                )

                for row in (
                    recording_candidate.itertuples(
                        index=False
                    )
                ):
                    cohort_candidate_rows.append({
                        "recording_id": (
                            row.recording_id
                        ),
                        "cohort": row.cohort,
                        "raw_cohort_value": (
                            row.raw_cohort_value
                        ),
                        "source_path": str(path),
                        "source_type": (
                            "participant_level"
                        ),
                        "cohort_column": (
                            cohort_column
                        ),
                        "priority": (
                            priority + 20
                        ),
                    })

                raw_values = sorted(
                    {
                        str(value)
                        for value in frame[
                            cohort_column
                        ].dropna().unique()
                    }
                )

                cohort_source_audit_rows.append({
                    "path": str(path),
                    "source_type": (
                        "participant_level"
                    ),
                    "cohort_column": (
                        cohort_column
                    ),
                    "priority": (
                        priority + 20
                    ),
                    "rows": len(frame),
                    "normalized_labels": (
                        len(
                            participant_candidate
                        )
                    ),
                    "population_coverage": (
                        recording_candidate[
                            "recording_id"
                        ].nunique()
                    ),
                    "raw_values": " | ".join(
                        raw_values
                    ),
                    "reason": "",
                })


cohort_candidates = pd.DataFrame(
    cohort_candidate_rows
)

cohort_source_audit = pd.DataFrame(
    cohort_source_audit_rows
)

export_dataframe(
    cohort_source_audit,
    LOG_DIR / "cohort_source_audit.csv",
)

display(
    cohort_source_audit.sort_values(
        [
            "population_coverage",
            "priority",
        ],
        ascending=[False, True],
    )
)


if cohort_candidates.empty:
    raise RuntimeError(
        "No valid ALS/control cohort labels were recovered. "
        "Review cohort_source_audit.csv."
    )


# ---------------------------------------------------------------------
# Check agreement across all usable sources
# ---------------------------------------------------------------------

cohort_conflict_counts = (
    cohort_candidates.groupby(
        "recording_id"
    )["cohort"]
    .nunique()
    .reset_index(
        name="distinct_cohort_labels"
    )
)

conflicting_recordings = (
    cohort_conflict_counts.loc[
        cohort_conflict_counts[
            "distinct_cohort_labels"
        ] > 1
    ]
)

if not conflicting_recordings.empty:
    conflict_detail = (
        cohort_candidates.loc[
            cohort_candidates[
                "recording_id"
            ].isin(
                conflicting_recordings[
                    "recording_id"
                ]
            )
        ]
        .sort_values(
            [
                "recording_id",
                "priority",
                "source_path",
            ]
        )
    )

    export_dataframe(
        conflict_detail,
        LOG_DIR
        / "metadata_cross_source_cohort_conflicts.csv",
    )

    display(conflict_detail)

    raise RuntimeError(
        "Conflicting cohort labels were found across "
        "metadata sources. Review "
        "metadata_cross_source_cohort_conflicts.csv."
    )


# Select the highest-priority valid source per recording.
cohort_map = (
    cohort_candidates.sort_values(
        [
            "recording_id",
            "priority",
            "source_type",
            "source_path",
        ]
    )
    .drop_duplicates(
        subset=["recording_id"],
        keep="first",
    )[
        [
            "recording_id",
            "cohort",
            "source_path",
            "source_type",
            "cohort_column",
            "raw_cohort_value",
            "priority",
        ]
    ]
    .rename(
        columns={
            "source_path": (
                "cohort_source_path"
            ),
            "source_type": (
                "cohort_source_type"
            ),
            "cohort_column": (
                "cohort_source_column"
            ),
            "raw_cohort_value": (
                "cohort_raw_value"
            ),
            "priority": (
                "cohort_source_priority"
            ),
        }
    )
)


# ---------------------------------------------------------------------
# Final governed population table
# ---------------------------------------------------------------------

frozen_population = pd.DataFrame({
    "recording_id": sorted(
        population_intersection
    )
})

frozen_population = (
    frozen_population
    .merge(
        participant_map,
        on="recording_id",
        how="left",
        validate="one_to_one",
    )
    .merge(
        cohort_map,
        on="recording_id",
        how="left",
        validate="one_to_one",
    )
)


missing_participant = (
    frozen_population[
        "participant_id"
    ].isna()
    | frozen_population[
        "participant_id"
    ].eq("")
)

missing_cohort = (
    frozen_population[
        "cohort"
    ].isna()
)

unexpected_cohorts = sorted(
    set(
        frozen_population[
            "cohort"
        ].dropna()
    )
    - {
        "ALS",
        "CONTROL",
    }
)

participant_cohort_consistency = (
    frozen_population.dropna(
        subset=[
            "participant_id",
            "cohort",
        ]
    )
    .groupby("participant_id")[
        "cohort"
    ]
    .nunique()
)

inconsistent_participants = (
    participant_cohort_consistency.loc[
        participant_cohort_consistency > 1
    ]
)

observed_cohort_counts = (
    frozen_population[
        "cohort"
    ]
    .value_counts()
    .to_dict()
)

metadata_resolution_audit = pd.DataFrame([
    {
        "check": "Frozen population rows",
        "observed": len(
            frozen_population
        ),
        "expected": (
            FINAL_POPULATION_N
        ),
        "passed": (
            len(frozen_population)
            == FINAL_POPULATION_N
        ),
    },
    {
        "check": "Participant count",
        "observed": (
            frozen_population[
                "participant_id"
            ].nunique()
        ),
        "expected": (
            EXPECTED_PARTICIPANTS
        ),
        "passed": (
            frozen_population[
                "participant_id"
            ].nunique()
            == EXPECTED_PARTICIPANTS
        ),
    },
    {
        "check": (
            "Missing participant IDs"
        ),
        "observed": int(
            missing_participant.sum()
        ),
        "expected": 0,
        "passed": (
            not missing_participant.any()
        ),
    },
    {
        "check": (
            "Missing cohort labels"
        ),
        "observed": int(
            missing_cohort.sum()
        ),
        "expected": 0,
        "passed": (
            not missing_cohort.any()
        ),
    },
    {
        "check": (
            "ALS recording count"
        ),
        "observed": (
            observed_cohort_counts.get(
                "ALS",
                0,
            )
        ),
        "expected": (
            EXPECTED_COHORT_RECORDINGS[
                "ALS"
            ]
        ),
        "passed": (
            observed_cohort_counts.get(
                "ALS",
                0,
            )
            == EXPECTED_COHORT_RECORDINGS[
                "ALS"
            ]
        ),
    },
    {
        "check": (
            "Control recording count"
        ),
        "observed": (
            observed_cohort_counts.get(
                "CONTROL",
                0,
            )
        ),
        "expected": (
            EXPECTED_COHORT_RECORDINGS[
                "CONTROL"
            ]
        ),
        "passed": (
            observed_cohort_counts.get(
                "CONTROL",
                0,
            )
            == EXPECTED_COHORT_RECORDINGS[
                "CONTROL"
            ]
        ),
    },
    {
        "check": (
            "Unexpected cohort labels"
        ),
        "observed": (
            " | ".join(
                unexpected_cohorts
            )
        ),
        "expected": "none",
        "passed": (
            not unexpected_cohorts
        ),
    },
    {
        "check": (
            "Participants with "
            "conflicting cohorts"
        ),
        "observed": len(
            inconsistent_participants
        ),
        "expected": 0,
        "passed": (
            inconsistent_participants.empty
        ),
    },
])


display(metadata_resolution_audit)

export_dataframe(
    metadata_resolution_audit,
    LOG_DIR
    / "metadata_resolution_audit.csv",
)


if not metadata_resolution_audit[
    "passed"
].all():
    unresolved = frozen_population.loc[
        missing_participant
        | missing_cohort
    ].copy()

    export_dataframe(
        unresolved,
        LOG_DIR
        / "metadata_unresolved_recordings.csv",
    )

    raise RuntimeError(
        "Frozen-population metadata resolution "
        "still failed. Review "
        "cohort_source_audit.csv, "
        "metadata_resolution_audit.csv, and "
        "metadata_unresolved_recordings.csv."
    )


print()
print(
    "Frozen population rows:",
    len(frozen_population),
)

print(
    "Participants:",
    frozen_population[
        "participant_id"
    ].nunique(),
)

display(
    frozen_population.groupby(
        "cohort"
    ).agg(
        recordings=(
            "recording_id",
            "nunique",
        ),
        participants=(
            "participant_id",
            "nunique",
        ),
    )
)

display(
    frozen_population[
        [
            "recording_id",
            "participant_id",
            "cohort",
            "cohort_source_type",
            "cohort_source_column",
            "cohort_raw_value",
        ]
    ].head()
)


export_dataframe(
    frozen_population,
    TABLE_DIR
    / "frozen_519_population_with_metadata.csv",
)

In [ ]:
# ---------------------------------------------------------------------
# CORRECTIVE CELL: resolve the remaining cohort labels from
# Diagnosis + diagnosis_resolution and rebuild the frozen population
# ---------------------------------------------------------------------

def normalize_cohort_complete(
    value: Any,
) -> str | float:
    """
    Normalize explicit diagnosis/cohort values, including provenance-style
    resolution labels such as confirmed_control_id_rule.

    Control is checked before ALS so that labels such as non_als are not
    incorrectly classified as ALS.
    """
    if value is None or pd.isna(value):
        return np.nan

    raw = str(value).strip()

    if raw.casefold() in {
        "",
        "nan",
        "none",
        "null",
        "na",
        "n/a",
        "unknown",
        "missing",
    }:
        return np.nan

    text = raw.casefold()
    compact = re.sub(r"[^a-z0-9]+", "", text)

    # Explicit control labels and resolution/provenance strings.
    control_tokens = [
        "control",
        "healthycontrol",
        "confirmedcontrol",
        "controlidrule",
        "nonals",
        "unaffected",
    ]

    if any(token in compact for token in control_tokens):
        return "CONTROL"

    # Explicit ALS labels and resolution/provenance strings.
    als_tokens = [
        "als",
        "amyotrophiclateralsclerosis",
        "motorneurondisease",
        "investigatorconfirmedals",
        "confirmedals",
    ]

    if any(token in compact for token in als_tokens):
        return "ALS"

    return np.nan


def cohort_source_priority(
    column_name: str,
    source_type: str,
) -> int | None:
    """
    Lower values receive precedence.

    Eligibility fields are excluded because they indicate admission to the
    analysis population, not ALS/control membership.
    """
    compact = re.sub(
        r"[^a-z0-9]+",
        "",
        str(column_name).casefold(),
    )

    if any(
        token in compact
        for token in [
            "eligible",
            "eligibility",
            "included",
            "excluded",
            "inclusion",
            "exclusion",
        ]
    ):
        return None

    if "diagnosisresolution" in compact:
        base_priority = 0
    elif "diagnosisanalysis" in compact:
        base_priority = 1
    elif compact in {
        "diagnosis",
        "cohort",
        "cohortlabel",
        "group",
        "diagnosticgroup",
        "clinicalgroup",
    }:
        base_priority = 2
    elif "diagnosis" in compact:
        base_priority = 3
    elif "cohort" in compact:
        base_priority = 4
    else:
        base_priority = 10

    # Prefer recording-level labels over participant-derived labels.
    return base_priority if source_type == "recording_level" else base_priority + 20


# ---------------------------------------------------------------------
# Collect every explicit cohort label from every usable metadata table
# ---------------------------------------------------------------------

cohort_candidate_rows = []
cohort_source_rows = []

for catalog_row in metadata_catalog.itertuples(index=False):

    cohort_column = getattr(catalog_row, "cohort_column", None)

    if not isinstance(cohort_column, str) or not cohort_column.strip():
        continue

    path = Path(catalog_row.path)

    id_column = getattr(catalog_row, "id_column", None)
    participant_column = getattr(
        catalog_row,
        "participant_column",
        None,
    )

    # -------------------------------------------------------------
    # Recording-level cohort source
    # -------------------------------------------------------------
    if isinstance(id_column, str) and id_column.strip():

        priority = cohort_source_priority(
            cohort_column,
            "recording_level",
        )

        if priority is not None:
            try:
                frame = read_metadata_table(
                    path,
                    usecols=[id_column, cohort_column],
                )
            except Exception as exc:
                cohort_source_rows.append({
                    "path": str(path),
                    "source_type": "recording_level",
                    "cohort_column": cohort_column,
                    "priority": priority,
                    "rows": np.nan,
                    "normalized_rows": 0,
                    "population_coverage": 0,
                    "raw_values": "",
                    "error": f"{type(exc).__name__}: {exc}",
                })
            else:
                candidate = pd.DataFrame({
                    "recording_id": (
                        frame[id_column]
                        .map(canonical_recording_id)
                    ),
                    "raw_cohort_value": frame[cohort_column],
                })

                candidate = candidate.loc[
                    candidate["recording_id"].isin(
                        population_intersection
                    )
                ].copy()

                candidate["cohort"] = candidate[
                    "raw_cohort_value"
                ].map(normalize_cohort_complete)

                resolved = candidate.loc[
                    candidate["cohort"].notna()
                ].copy()

                for row in resolved.itertuples(index=False):
                    cohort_candidate_rows.append({
                        "recording_id": row.recording_id,
                        "cohort": row.cohort,
                        "raw_cohort_value": row.raw_cohort_value,
                        "source_path": str(path),
                        "source_type": "recording_level",
                        "cohort_column": cohort_column,
                        "priority": priority,
                    })

                cohort_source_rows.append({
                    "path": str(path),
                    "source_type": "recording_level",
                    "cohort_column": cohort_column,
                    "priority": priority,
                    "rows": len(frame),
                    "normalized_rows": len(resolved),
                    "population_coverage": (
                        resolved["recording_id"].nunique()
                    ),
                    "raw_values": " | ".join(
                        sorted(
                            {
                                str(value)
                                for value in candidate[
                                    "raw_cohort_value"
                                ].dropna().unique()
                            }
                        )
                    ),
                    "error": "",
                })

    # -------------------------------------------------------------
    # Participant-level cohort source
    # -------------------------------------------------------------
    if (
        isinstance(participant_column, str)
        and participant_column.strip()
    ):

        priority = cohort_source_priority(
            cohort_column,
            "participant_level",
        )

        if priority is not None:
            try:
                frame = read_metadata_table(
                    path,
                    usecols=[
                        participant_column,
                        cohort_column,
                    ],
                )
            except Exception as exc:
                cohort_source_rows.append({
                    "path": str(path),
                    "source_type": "participant_level",
                    "cohort_column": cohort_column,
                    "priority": priority,
                    "rows": np.nan,
                    "normalized_rows": 0,
                    "population_coverage": 0,
                    "raw_values": "",
                    "error": f"{type(exc).__name__}: {exc}",
                })
            else:
                participant_candidate = pd.DataFrame({
                    "participant_id": (
                        frame[participant_column]
                        .map(canonical_participant_id)
                    ),
                    "raw_cohort_value": frame[cohort_column],
                })

                participant_candidate["cohort"] = (
                    participant_candidate[
                        "raw_cohort_value"
                    ].map(normalize_cohort_complete)
                )

                participant_candidate = (
                    participant_candidate.loc[
                        participant_candidate[
                            "participant_id"
                        ].ne("")
                        & participant_candidate[
                            "cohort"
                        ].notna()
                    ]
                    .copy()
                )

                # A source is invalid if one participant has two cohort labels.
                conflicts = (
                    participant_candidate.groupby(
                        "participant_id"
                    )["cohort"]
                    .nunique()
                )

                conflicts = conflicts.loc[conflicts > 1]

                if not conflicts.empty:
                    conflict_detail = (
                        participant_candidate.loc[
                            participant_candidate[
                                "participant_id"
                            ].isin(conflicts.index)
                        ]
                        .sort_values(
                            ["participant_id", "cohort"]
                        )
                    )

                    export_dataframe(
                        conflict_detail,
                        LOG_DIR
                        / (
                            "participant_cohort_conflict_"
                            + re.sub(
                                r"[^A-Za-z0-9]+",
                                "_",
                                path.stem,
                            )
                            + ".csv"
                        ),
                    )

                    continue

                participant_candidate = (
                    participant_candidate
                    .drop_duplicates(
                        subset=["participant_id"],
                        keep="first",
                    )
                )

                recording_candidate = participant_map.merge(
                    participant_candidate,
                    on="participant_id",
                    how="inner",
                    validate="many_to_one",
                )

                for row in recording_candidate.itertuples(
                    index=False
                ):
                    cohort_candidate_rows.append({
                        "recording_id": row.recording_id,
                        "cohort": row.cohort,
                        "raw_cohort_value": row.raw_cohort_value,
                        "source_path": str(path),
                        "source_type": "participant_level",
                        "cohort_column": cohort_column,
                        "priority": priority,
                    })

                cohort_source_rows.append({
                    "path": str(path),
                    "source_type": "participant_level",
                    "cohort_column": cohort_column,
                    "priority": priority,
                    "rows": len(frame),
                    "normalized_rows": len(
                        participant_candidate
                    ),
                    "population_coverage": (
                        recording_candidate[
                            "recording_id"
                        ].nunique()
                    ),
                    "raw_values": " | ".join(
                        sorted(
                            {
                                str(value)
                                for value in frame[
                                    cohort_column
                                ].dropna().unique()
                            }
                        )
                    ),
                    "error": "",
                })


cohort_candidates = pd.DataFrame(cohort_candidate_rows)
cohort_source_audit = pd.DataFrame(cohort_source_rows)

export_dataframe(
    cohort_source_audit,
    LOG_DIR / "cohort_source_audit_CORRECTED.csv",
)

display(
    cohort_source_audit.sort_values(
        ["population_coverage", "priority"],
        ascending=[False, True],
    )
)

if cohort_candidates.empty:
    raise RuntimeError(
        "No explicit ALS/control labels were recovered."
    )


# ---------------------------------------------------------------------
# Require agreement among all metadata sources
# ---------------------------------------------------------------------

cross_source_conflicts = (
    cohort_candidates.groupby("recording_id")["cohort"]
    .nunique()
    .reset_index(name="distinct_cohort_labels")
)

cross_source_conflicts = cross_source_conflicts.loc[
    cross_source_conflicts["distinct_cohort_labels"] > 1
]

if not cross_source_conflicts.empty:

    conflict_detail = (
        cohort_candidates.loc[
            cohort_candidates["recording_id"].isin(
                cross_source_conflicts["recording_id"]
            )
        ]
        .sort_values(
            [
                "recording_id",
                "priority",
                "source_path",
            ]
        )
    )

    export_dataframe(
        conflict_detail,
        LOG_DIR
        / "metadata_cross_source_cohort_conflicts_CORRECTED.csv",
    )

    display(conflict_detail)

    raise RuntimeError(
        "Conflicting cohort labels were found across metadata "
        "sources. Review the exported conflict table."
    )


# Select the most authoritative available source for each recording.
cohort_map = (
    cohort_candidates.sort_values(
        [
            "recording_id",
            "priority",
            "source_type",
            "source_path",
        ]
    )
    .drop_duplicates(
        subset=["recording_id"],
        keep="first",
    )
    .rename(
        columns={
            "source_path": "cohort_source_path",
            "source_type": "cohort_source_type",
            "cohort_column": "cohort_source_column",
            "raw_cohort_value": "cohort_raw_value",
            "priority": "cohort_source_priority",
        }
    )
)


# ---------------------------------------------------------------------
# Rebuild the final 519-recording population from scratch
# ---------------------------------------------------------------------

frozen_population = pd.DataFrame({
    "recording_id": sorted(population_intersection)
})

frozen_population = (
    frozen_population
    .merge(
        participant_map,
        on="recording_id",
        how="left",
        validate="one_to_one",
    )
    .merge(
        cohort_map[
            [
                "recording_id",
                "cohort",
                "cohort_source_path",
                "cohort_source_type",
                "cohort_source_column",
                "cohort_raw_value",
                "cohort_source_priority",
            ]
        ],
        on="recording_id",
        how="left",
        validate="one_to_one",
    )
)


# ---------------------------------------------------------------------
# Blocking validation
# ---------------------------------------------------------------------

observed_counts = (
    frozen_population["cohort"]
    .value_counts(dropna=False)
    .to_dict()
)

missing_participant = (
    frozen_population["participant_id"].isna()
    | frozen_population["participant_id"].eq("")
)

missing_cohort = frozen_population["cohort"].isna()

participant_cohort_counts = (
    frozen_population.dropna(
        subset=["participant_id", "cohort"]
    )
    .groupby("participant_id")["cohort"]
    .nunique()
)

inconsistent_participants = participant_cohort_counts.loc[
    participant_cohort_counts > 1
]

metadata_resolution_audit = pd.DataFrame([
    {
        "check": "Frozen population rows",
        "observed": len(frozen_population),
        "expected": 519,
        "passed": len(frozen_population) == 519,
    },
    {
        "check": "Participant count",
        "observed": (
            frozen_population["participant_id"].nunique()
        ),
        "expected": 224,
        "passed": (
            frozen_population["participant_id"].nunique()
            == 224
        ),
    },
    {
        "check": "Missing participant IDs",
        "observed": int(missing_participant.sum()),
        "expected": 0,
        "passed": not missing_participant.any(),
    },
    {
        "check": "Missing cohort labels",
        "observed": int(missing_cohort.sum()),
        "expected": 0,
        "passed": not missing_cohort.any(),
    },
    {
        "check": "ALS recording count",
        "observed": observed_counts.get("ALS", 0),
        "expected": 418,
        "passed": observed_counts.get("ALS", 0) == 418,
    },
    {
        "check": "Control recording count",
        "observed": observed_counts.get("CONTROL", 0),
        "expected": 101,
        "passed": (
            observed_counts.get("CONTROL", 0) == 101
        ),
    },
    {
        "check": "Participants with conflicting cohorts",
        "observed": len(inconsistent_participants),
        "expected": 0,
        "passed": inconsistent_participants.empty,
    },
])

display(metadata_resolution_audit)

export_dataframe(
    metadata_resolution_audit,
    LOG_DIR / "metadata_resolution_audit_CORRECTED.csv",
)

if not metadata_resolution_audit["passed"].all():

    unresolved = frozen_population.loc[
        missing_participant | missing_cohort
    ].copy()

    export_dataframe(
        unresolved,
        LOG_DIR / "metadata_unresolved_recordings_CORRECTED.csv",
    )

    raise RuntimeError(
        "Corrected cohort resolution did not reproduce the governed "
        "519-recording population. Review the corrected audit tables."
    )


print("\nCORRECTED METADATA RESOLUTION: PASS")

display(
    frozen_population.groupby("cohort").agg(
        recordings=("recording_id", "nunique"),
        participants=("participant_id", "nunique"),
    )
)

display(
    frozen_population[
        [
            "recording_id",
            "participant_id",
            "cohort",
            "cohort_source_column",
            "cohort_raw_value",
        ]
    ].head(20)
)

export_dataframe(
    frozen_population,
    TABLE_DIR / "frozen_519_population_with_metadata.csv",
)

## 11. Merge production annotations to the frozen population

In [ ]:
annotation_ratings = rating_base.merge(
    frozen_population,
    on="recording_id",
    how="left",
    indicator=True,
)

population_merge_summary = pd.DataFrame([
    {
        "metric": "unique_annotation_recordings",
        "value": rating_base["recording_id"].nunique(),
    },
    {
        "metric": "unique_annotation_ratings",
        "value": rating_base["recording_rating_id"].nunique(),
    },
    {
        "metric": "annotation_recordings_in_frozen_519",
        "value": annotation_ratings.loc[
            annotation_ratings["_merge"] == "both",
            "recording_id",
        ].nunique(),
    },
    {
        "metric": "annotation_recordings_outside_frozen_519",
        "value": annotation_ratings.loc[
            annotation_ratings["_merge"] == "left_only",
            "recording_id",
        ].nunique(),
    },
    {
        "metric": "frozen_519_without_annotation",
        "value": len(
            set(frozen_population["recording_id"])
            - set(annotation_ratings.loc[annotation_ratings["_merge"] == "both", "recording_id"])
        ),
    },
    {
        "metric": "frozen_population_expected",
        "value": FINAL_POPULATION_N,
    },
])

unmatched_annotations = annotation_ratings.loc[
    annotation_ratings["_merge"] == "left_only"
].copy()
unannotated_population = frozen_population.loc[
    ~frozen_population["recording_id"].isin(annotation_ratings["recording_id"])
].copy()

display(population_merge_summary)
display(unmatched_annotations.head())
display(unannotated_population.head())

export_dataframe(population_merge_summary, TABLE_DIR / "humanqc_population_merge_summary.csv")
export_dataframe(unmatched_annotations, LOG_DIR / "humanqc_annotations_outside_frozen_population.csv")
export_dataframe(unannotated_population, LOG_DIR / "humanqc_frozen_population_without_annotation.csv")

# Attach metadata to summary tables.
metadata_columns = ["recording_id", "participant_id", "cohort"]
recording_item_analysis = recording_item_summary.merge(
    frozen_population[metadata_columns],
    on="recording_id",
    how="inner",
)
recording_source_family_analysis = recording_source_family_summary.merge(
    frozen_population[metadata_columns],
    on="recording_id",
    how="inner",
)
recording_signal_family_analysis = recording_signal_family_summary.merge(
    frozen_population[metadata_columns],
    on="recording_id",
    how="inner",
)

export_dataframe(recording_item_analysis, TABLE_DIR / "humanqc_recording_item_analysis_population.csv")
export_dataframe(
    recording_source_family_analysis,
    TABLE_DIR / "humanqc_recording_source_family_analysis_population.csv",
)
export_dataframe(
    recording_signal_family_analysis,
    TABLE_DIR / "humanqc_recording_signal_family_analysis_population.csv",
)


## 12. Prevalence and rater-allocation audit

In [ ]:
def prevalence_table(
    frame: pd.DataFrame,
    group_columns: list[str],
) -> pd.DataFrame:
    return (
        frame.groupby(group_columns, dropna=False)
        .agg(
            ratings=("recording_rating_id", "nunique"),
            recordings=("recording_id", "nunique"),
            participants=("participant_id", "nunique"),
            positives=("present", "sum"),
            positive_recordings=("recording_id", lambda values: values[frame.loc[values.index, "present"]].nunique()),
            positive_participants=("participant_id", lambda values: values[frame.loc[values.index, "present"]].nunique()),
            median_positive_duration_fraction=(
                "duration_fraction",
                lambda values: float(values[values > 0].median()) if (values > 0).any() else np.nan,
            ),
        )
        .reset_index()
        .assign(
            prevalence_percent=lambda table: 100 * table["positives"] / table["ratings"]
        )
    )


family_prevalence_by_rater = prevalence_table(
    recording_signal_family_analysis,
    ["signal_family", "rater"],
)
family_prevalence_by_cohort = prevalence_table(
    recording_signal_family_analysis,
    ["signal_family", "cohort"],
)
item_prevalence_by_rater = prevalence_table(
    recording_item_analysis,
    ["source_family", "item", "rater"],
)
item_prevalence_overall = prevalence_table(
    recording_item_analysis,
    ["source_family", "signal_family", "item"],
).sort_values("prevalence_percent", ascending=False)

rater_allocation = (
    annotation_ratings.loc[annotation_ratings["_merge"] == "both"]
    .groupby(["rater", "cohort"], dropna=False)
    .agg(
        ratings=("recording_rating_id", "nunique"),
        recordings=("recording_id", "nunique"),
        participants=("participant_id", "nunique"),
    )
    .reset_index()
)

duplicate_rater_cohort = (
    annotation_ratings.loc[
        (annotation_ratings["_merge"] == "both")
        & annotation_ratings["recording_id"].isin(
            duplicate_recordings.loc[
                duplicate_recordings["multiple_raters"],
                "recording_id",
            ]
        )
    ]
    .sort_values(["recording_id", "rater"])
)

display(family_prevalence_by_rater)
display(family_prevalence_by_cohort)
display(item_prevalence_overall.head(30))
display(rater_allocation)

export_dataframe(family_prevalence_by_rater, TABLE_DIR / "humanqc_family_prevalence_by_rater.csv")
export_dataframe(family_prevalence_by_cohort, TABLE_DIR / "humanqc_family_prevalence_by_cohort.csv")
export_dataframe(item_prevalence_by_rater, TABLE_DIR / "humanqc_item_prevalence_by_rater.csv")
export_dataframe(item_prevalence_overall, TABLE_DIR / "humanqc_item_prevalence_overall.csv")
export_dataframe(rater_allocation, TABLE_DIR / "humanqc_rater_allocation_by_cohort.csv")
export_dataframe(duplicate_rater_cohort, LOG_DIR / "humanqc_duplicate_rating_metadata.csv")


## 13. Human-outcome support classification for the proposed construct map

This classification is based on the human-reference side only. Joint support with each signal-derived indicator is intentionally deferred until the exact production annotation set and interval merge tolerance are approved.


In [ ]:
def outcome_support(
    outcome_level: str,
    outcome_value: str,
) -> pd.DataFrame:
    if outcome_level == "signal_family":
        frame = recording_signal_family_analysis.loc[
            recording_signal_family_analysis["signal_family"] == outcome_value
        ].copy()
    elif outcome_level == "source_family":
        frame = recording_source_family_analysis.loc[
            recording_source_family_analysis["source_family"] == outcome_value
        ].copy()
    elif outcome_level == "item":
        frame = recording_item_analysis.loc[
            recording_item_analysis["item"] == outcome_value
        ].copy()
    else:
        raise ValueError(outcome_level)

    return frame


support_rows = []

for signal_family, outcome_level, outcome_value, display_name in PROPOSED_HUMAN_OUTCOMES:
    frame = outcome_support(outcome_level, outcome_value)
    positives = frame.loc[frame["present"]].copy()

    positive_counts_by_rater = positives["rater"].value_counts()
    maximum_rater_share = (
        float(positive_counts_by_rater.max() / len(positives))
        if len(positives)
        else np.nan
    )

    n_positive = int(frame["present"].sum())
    n_negative = int((~frame["present"]).sum())
    n_positive_participants = int(positives["participant_id"].nunique())
    n_positive_raters = int(positives["rater"].nunique())

    all_confirmatory_gates = bool(
        n_positive >= 20
        and n_negative >= 20
        and n_positive_participants >= 10
        and n_positive_raters >= 2
        and (np.isnan(maximum_rater_share) or maximum_rater_share <= 0.80)
        and len(frame) >= 50
    )

    if all_confirmatory_gates:
        support_class = "confirmatory_human_support"
    elif n_positive >= 10 and n_negative >= 10:
        support_class = "exploratory_human_support"
    else:
        support_class = "descriptive_only"

    support_rows.append({
        "signal_family": signal_family,
        "outcome_level": outcome_level,
        "outcome_value": outcome_value,
        "display_name": display_name,
        "ratings": len(frame),
        "recordings": frame["recording_id"].nunique(),
        "participants": frame["participant_id"].nunique(),
        "positive_ratings": n_positive,
        "negative_ratings": n_negative,
        "positive_recordings": positives["recording_id"].nunique(),
        "positive_participants": n_positive_participants,
        "positive_raters": n_positive_raters,
        "maximum_positive_rater_share": maximum_rater_share,
        "support_class": support_class,
        "joint_signal_feature_support_pending": True,
    })

human_outcome_support = pd.DataFrame(support_rows)
display(human_outcome_support)

export_dataframe(
    human_outcome_support,
    TABLE_DIR / "humanqc_proposed_outcome_support_classification.csv",
)


## 14. Stage-1 audit figures

In [ ]:
mpl.rcParams.update({
    "font.family": ["Arial", "DejaVu Sans"],
    "font.size": 8,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})

# Figure 1: annotation inventory and cohort allocation.
fig, axes = plt.subplots(1, 2, figsize=(7.15, 3.25), constrained_layout=True)

folder_plot = folder_summary.set_index("rater").reindex(EXPECTED_RATERS)
axes[0].bar(folder_plot.index, folder_plot["csv_files"])
axes[0].set_ylabel("CSV files")
axes[0].set_title("A  Annotation inventory", loc="left", fontweight="bold")
axes[0].tick_params(axis="x", rotation=30)

allocation_pivot = (
    rater_allocation.pivot(index="rater", columns="cohort", values="recordings")
    .reindex(EXPECTED_RATERS)
    .fillna(0)
)
allocation_pivot.plot(kind="bar", ax=axes[1])
axes[1].set_ylabel("Frozen-population recordings")
axes[1].set_title("B  Rater allocation by cohort", loc="left", fontweight="bold")
axes[1].legend(frameon=False)
axes[1].tick_params(axis="x", rotation=30)

for suffix in [".png", ".svg", ".pdf"]:
    fig.savefig(FIGURE_DIR / f"Supplementary_annotation_inventory{suffix}", dpi=400 if suffix == ".png" else None)
plt.show()

# Figure 2: family prevalence by rater.
family_order = ["QADD", "QGAIN", "QREV", "QCHAN", "QDIST", "QTEMP", "TASK"]
prevalence_matrix = (
    family_prevalence_by_rater.pivot(
        index="signal_family",
        columns="rater",
        values="prevalence_percent",
    )
    .reindex(index=family_order, columns=EXPECTED_RATERS)
)

fig, ax = plt.subplots(figsize=(5.7, 3.8), constrained_layout=True)
image = ax.imshow(prevalence_matrix.to_numpy(), aspect="auto")
ax.set_xticks(np.arange(len(prevalence_matrix.columns)))
ax.set_xticklabels(prevalence_matrix.columns)
ax.set_yticks(np.arange(len(prevalence_matrix.index)))
ax.set_yticklabels(prevalence_matrix.index)
ax.set_title("Human-label prevalence by family and rater", loc="left", fontweight="bold")
colorbar = fig.colorbar(image, ax=ax)
colorbar.set_label("Positive ratings (%)")

for suffix in [".png", ".svg", ".pdf"]:
    fig.savefig(FIGURE_DIR / f"Supplementary_family_prevalence_by_rater{suffix}", dpi=400 if suffix == ".png" else None)
plt.show()

# Figure 3: merge sensitivity.
fig, ax = plt.subplots(figsize=(5.4, 3.2), constrained_layout=True)
ax.plot(
    1000 * merge_sensitivity_summary["merge_tolerance_sec"],
    merge_sensitivity_summary["total_merged_events"],
    marker="o",
)
ax.set_xlabel("Interval merge tolerance (ms)")
ax.set_ylabel("Total merged item events")
ax.set_title("Interval-count sensitivity to merge tolerance", loc="left", fontweight="bold")
ax.grid(axis="y", alpha=0.25)

for suffix in [".png", ".svg", ".pdf"]:
    fig.savefig(FIGURE_DIR / f"Supplementary_interval_merge_sensitivity{suffix}", dpi=400 if suffix == ".png" else None)
plt.show()


## 15. Run manifest and blocking completion checklist

In [ ]:
blocking_checks = pd.DataFrame([
    {
        "check": "Expected rater folders found",
        "passed": set(EXPECTED_RATERS).issubset(set(folder_summary["rater"])),
        "detail": f"found={sorted(folder_summary['rater'].unique())}",
    },
    {
        "check": "All CSV files loaded",
        "passed": bool(file_load_audit["loaded"].all()),
        "detail": f"files={len(file_load_audit)}",
    },
    {
        "check": "No missing required columns",
        "passed": not file_load_audit["missing_required_columns"].fillna("").ne("").any(),
        "detail": "",
    },
    {
        "check": "No within-rater duplicate ratings",
        "passed": within_rater_duplicates.empty,
        "detail": f"duplicates={len(within_rater_duplicates)}",
    },
    {
        "check": "No unresolved JSON parse issues",
        "passed": parse_issues.empty,
        "detail": f"issues={len(parse_issues)}",
    },
    {
        "check": "No unknown/missing taxonomy keys",
        "passed": taxonomy_issues.empty,
        "detail": f"issues={len(taxonomy_issues)}",
    },
    {
        "check": "Frozen population reconstructed",
        "passed": len(frozen_population) == FINAL_POPULATION_N,
        "detail": f"rows={len(frozen_population)}",
    },
    {
        "check": "Stage 1 contains no correspondence models",
        "passed": not ALLOW_CORRESPONDENCE_MODELS,
        "detail": "audit only",
    },
])

display(blocking_checks)
export_dataframe(blocking_checks, LOG_DIR / "stage1_blocking_completion_checklist.csv")

manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "human_qc_root": str(HUMAN_QC_ROOT),
    "metadata_root": str(METADATA_ROOT),
    "family_freeze_root": str(FAMILY_FREEZE_ROOT),
    "analysis_root": str(ANALYSIS_ROOT),
    "expected_raters": EXPECTED_RATERS,
    "csv_files": len(file_inventory),
    "unique_annotation_recordings": int(rating_base["recording_id"].nunique()),
    "unique_annotation_ratings": int(rating_base["recording_rating_id"].nunique()),
    "multiple_rater_recordings": int(duplicate_recordings["multiple_raters"].sum()),
    "raw_intervals": len(raw_interval_ledger),
    "valid_intervals": int(raw_interval_ledger["valid_interval"].sum()),
    "exact_duplicate_intervals": int(raw_interval_ledger["exact_duplicate"].sum()),
    "parse_issues": len(parse_issues),
    "taxonomy_issues": len(taxonomy_issues),
    "provisional_merge_tolerance_sec": PROVISIONAL_MERGE_TOLERANCE_SEC,
    "merge_sensitivity_tolerances_sec": MERGE_SENSITIVITY_TOLERANCES_SEC,
    "frozen_population_n": len(frozen_population),
    "annotation_recordings_in_frozen_population": int(
        annotation_ratings.loc[
            annotation_ratings["_merge"] == "both",
            "recording_id",
        ].nunique()
    ),
    "models_allowed": ALLOW_CORRESPONDENCE_MODELS,
    "python": sys.version,
    "platform": platform.platform(),
}

(MANIFEST_DIR / "stage1_run_manifest.json").write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8",
)

if not blocking_checks["passed"].all():
    print(
        "\nSTAGE 1 COMPLETED WITH BLOCKING AUDIT ITEMS. "
        "Do not proceed to correspondence modeling until reviewed."
    )
else:
    print(
        "\nSTAGE 1 COMPLETED. "
        "The annotation tables are ready for scientific review, but model fitting remains disabled."
    )


---

## Stage-1 stop

Do **not** add or run correspondence models yet.

The following must be reviewed first:

1. recordings duplicated across raters;
2. annotation coverage of the frozen 519-recording population;
3. rater assignment by cohort;
4. JSON/taxonomy and segment-geometry issues;
5. adjacency-gap distribution and merge-tolerance sensitivity;
6. family/item prevalence by rater and cohort;
7. confirmatory, exploratory, and descriptive-only human outcomes.

After these are approved, Stage 2 will freeze the exact production annotation set, interval merge rule, construct-matched indicator map, and Bayesian model specification.
